# ORCA Veterinary Claims — T4 GPU PaddleOCR-VL + BPI Rules (v4.8)

Google Colab POC using **PyMuPDF → PaddleOCR-VL-1.5 on the T4 GPU → transparent BPI rules → deterministic Python validation → JSON/CSV**. The Paddle GPU runtime is the required primary path. A CPU-only PyTorch dependency is installed only to satisfy optional PaddleX/ModelScope imports without loading PyTorch's incompatible CUDA/NCCL libraries into the Paddle GPU process. **Qwen3-4B is retained as an optional fallback but disabled by default**; it is not required for OCR or exports. No live Orca actions.

**Runtime target:** Google Colab Python 3.13 + Tesla T4. This notebook checks CUDA Paddle, performs a real GPU tensor operation, checks PyTorch's CPU-only status, imports PaddleOCR-VL, and fails early with actionable diagnostics if any required check fails. A successful preflight verifies the runtime; first-time model download and inference still need to complete in the user's Colab session.


## Integrated workflow and runtime policy

1. Upload multiple PDFs/images.
2. Read/render every page with PyMuPDF and preserve native PDF text.
3. Run **PaddleOCR-VL-1.5 on `gpu:0`** on each page; retain raw OCR/layout output.
4. Classify using the eight BPI labels with explicit, auditable Python rules.
5. Extract metadata for every document class and genuine invoice-style rows.
6. Validate amounts and prepare pet-wise groups; ambiguous evidence, missing quantities, unclear pet assignment, and mismatches require human review.
7. Export JSON/CSV. No live Orca actions.

**GPU compatibility design:** PaddlePaddle owns the T4 GPU for OCR. PyTorch is installed from the CPU wheel index because PaddleX/ModelScope may import PyTorch during PaddleOCR-VL initialization; this prevents the observed `ncclCommWindowDeregister` failure from PyTorch's bundled CUDA/NCCL library. PyTorch is not used for the primary OCR inference.

**Qwen:** `QWEN_MODE = "off"` and `QWEN_LINE_ITEM_FALLBACK = False` by default. Qwen3-4B is not loaded into the T4 during the OCR run. If enabled in this same runtime, the notebook checks CUDA availability and records a skip/review status rather than breaking OCR/export. Qwen can be run later in a separate compatible runtime or through a separately configured inference endpoint.

This is a proof of concept. No OCR system can guarantee correct extraction for every unfamiliar layout; review flagged results before claim submission.


## 1. Install dependencies


In [1]:
# Colab package setup — run once in a fresh T4 runtime.
# Do not import paddle, torch, paddlex, or paddleocr before this cell.
# PaddleOCR-VL uses PaddlePaddle for GPU inference. PyTorch is CPU-only here
# because ModelScope/PaddleX may import torch as a dependency and the current
# Colab CUDA torch wheel fails to resolve ncclCommWindowDeregister.
import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment configured. Run the installation cell next.")


Environment configured. Run the installation cell next.


In [2]:
# Install Paddle GPU + PaddleOCR-VL stack first.
# Paddle 3.3.1 CUDA 12.6 wheel works with the Colab T4 driver via CUDA compatibility.
%pip install -q --no-cache-dir "paddlepaddle-gpu==3.3.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q --no-cache-dir "pandas==2.2.3" "Pillow<13" pymupdf "paddleocr==3.7.0" "paddlex[ocr]==3.7.2" transformers accelerate

# Replace CUDA-enabled torch (if present) with the CPU wheel. This avoids loading
# torch/lib/libtorch_cuda.so and its bundled NCCL into the primary Paddle process.
%pip uninstall -y -q torch torchvision torchaudio
%pip install -q --no-cache-dir "torch==2.11.0" --index-url https://download.pytorch.org/whl/cpu

print("Install commands completed. IMPORTANT: Runtime > Restart session, then run cells from the top.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.30 requires torchvision, which is not installed.
fastai 2.8.12 requires torchvision>=0.11, which is not installed.
Install commands completed. IMPORTANT: Runtime > Restart session, then run cells from the top.


In [1]:
import paddle

print("Paddle version:", paddle.__version__)
print("CUDA compiled:", paddle.is_compiled_with_cuda())

paddle.set_device("gpu:0")

x = paddle.randn([1000, 1000])
y = paddle.matmul(x, x)

print("Active device:", paddle.device.get_device())
print("GPU computation:", y.shape)

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Paddle version: 3.3.1
CUDA compiled: True
Active device: gpu:0
GPU computation: paddle.Size([1000, 1000])


In [2]:
from paddleocr import PaddleOCRVL

print("PaddleOCR-VL import successful")

PaddleOCR-VL import successful


## 2. Upload invoices and initialize folders

Supported: PDF, PNG, JPG/JPEG, TIFF, WEBP. Upload multiple documents at once.

In [3]:
from pathlib import Path
import os, re, json, time, zipfile, hashlib, logging
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import pandas as pd
import fitz
from PIL import Image
from google.colab import files

ROOT=Path('/content/orca_vet_line_items')
INPUT_DIR=ROOT/'inputs'; OUTPUT_DIR=ROOT/'outputs'; RAW_DIR=OUTPUT_DIR/'raw_ocr'
for d in (INPUT_DIR,OUTPUT_DIR,RAW_DIR): d.mkdir(parents=True,exist_ok=True)
logging.basicConfig(level=logging.INFO,format='%(levelname)s: %(message)s')
logger=logging.getLogger('orca-line-items')
uploaded=files.upload()
for name,data in uploaded.items():
    if Path(name).suffix.lower() in {'.pdf','.png','.jpg','.jpeg','.tif','.tiff','.webp'}:
        (INPUT_DIR/Path(name).name).write_bytes(data)
    else: logger.warning('Ignored non-document file: %s',name)
input_files=sorted(p for p in INPUT_DIR.iterdir() if p.is_file())
print('Documents:',len(input_files))
if not input_files: raise RuntimeError('No supported PDF/image documents uploaded.')

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Saving 23026778.pdf to 23026778.pdf
Documents: 1


In [4]:
# Required runtime preflight: Paddle GPU must work; PyTorch is intentionally CPU-only.
import sys, subprocess
import torch
import paddle
import paddlex
import paddleocr
from paddleocr import PaddleOCRVL

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA available (expected False):", torch.cuda.is_available())
print("PaddlePaddle:", paddle.__version__)
print("PaddleX:", getattr(paddlex, "__version__", "version unavailable"))
print("PaddleOCR:", getattr(paddleocr, "__version__", "version unavailable"))
print("PaddleOCRVL class:", PaddleOCRVL)
print("Paddle compiled with CUDA:", paddle.is_compiled_with_cuda())

if torch.cuda.is_available():
    raise RuntimeError("Unexpected CUDA-enabled PyTorch detected. Use the CPU-only PyTorch wheel in the install cell.")
if not paddle.is_compiled_with_cuda():
    raise RuntimeError("PaddlePaddle is CPU-only. Select Runtime > Change runtime type > T4 GPU, rerun install cell, restart session, and run all cells.")

paddle.set_device("gpu:0")
assert paddle.device.get_device() == "gpu:0", paddle.device.get_device()
# Real Paddle GPU operation, not just a CUDA availability flag.
x = paddle.randn([1024, 1024], dtype="float32")
y = paddle.matmul(x, x)
paddle.device.cuda.synchronize()
assert str(y.place).lower().find("gpu") >= 0, y.place
print("Paddle active device:", paddle.device.get_device())
print("Paddle GPU computation:", tuple(y.shape), "on", y.place)
try:
    print(subprocess.check_output(["nvidia-smi"], text=True)[:2500])
except Exception as exc:
    print("nvidia-smi diagnostic unavailable:", repr(exc))

print("PREFLIGHT PASSED: Paddle is executing on the T4; PyTorch is CPU-only by design.")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
PyTorch CUDA available (expected False): False
PaddlePaddle: 3.3.1
PaddleX: 3.7.2
PaddleOCR: 3.7.0
PaddleOCRVL class: <class 'paddleocr._pipelines.paddleocr_vl.PaddleOCRVL'>
Paddle compiled with CUDA: True
Paddle active device: gpu:0
Paddle GPU computation: (1024, 1024) on Place(gpu:0)
Wed Oct 28 11:52:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+=======

/tmp/ipykernel_3858/1023065025.py:28: VisibleDeprecationWarning: 
API "paddle.device.cuda.synchronize" is deprecated since 2.5.0, and will be removed in future versions. Please use "paddle.device.synchronize" instead.
    Reason: synchronize in paddle.device.cuda will be removed in future 
  paddle.device.cuda.synchronize()


## 3. PyMuPDF page ingestion

Native PDF text is preserved; pages are rendered for visual OCR. Images are treated as one-page documents.

In [5]:
def load_pages(path,dpi=200):
    pages=[]
    if path.suffix.lower()=='.pdf':
        doc=fitz.open(path)
        try:
            for n,page in enumerate(doc,1):
                pix=page.get_pixmap(matrix=fitz.Matrix(dpi/72,dpi/72),alpha=False)
                img=Image.frombytes('RGB',[pix.width,pix.height],pix.samples)
                pages.append({'source_file':path.name,'page_number':n,'native_text':page.get_text('text') or '', 'image':img,'page_count':len(doc)})
        finally: doc.close()
    else:
        with Image.open(path) as im: img=im.convert('RGB').copy()
        pages.append({'source_file':path.name,'page_number':1,'native_text':'','image':img,'page_count':1})
    return pages
all_pages=[]
for f in input_files:
    try: all_pages.extend(load_pages(f))
    except Exception as e: logger.exception('Cannot read %s: %s',f.name,e)
print('Pages:',len(all_pages))
if not all_pages: raise RuntimeError('No readable pages found.')

Pages: 1


## BPI classification rules — implementation notes

The supplied BPI rules are the authority for class decisions. The notebook implements a transparent, checkpoint-free rule engine. Eight canonical labels are used: `CLAIM_FORM`, `INVOICE`, `MEDICAL_RECORD`, `ESTIMATE`, `PHARMACY`, `ONLINE_PROVIDER`, `INCOMPLETE`, and `OTHER_UNKNOWN`.

- Claim form and invoice together on the same page → `INVOICE`.
- Completed veterinary bill → `INVOICE`. Medical records with complete billing used as the bill → `INVOICE`.
- Pharmacy receipts/prescriptions → `PHARMACY`.
- Online provider receipts/invoices → `ONLINE_PROVIDER` (display label: **Online Store**).
- Planned/future treatment, estimate/draft/treatment plan, or LOW/HIGH pricing → `ESTIMATE`.
- Clinical record without complete billing → `MEDICAL_RECORD`.
- Payment slip without invoice or explicit missing pages → `INCOMPLETE`.
- Ambiguous/conflicting or unsupported evidence → `OTHER_UNKNOWN` and human review.

The classifier uses only evidence visible in native PDF text and PaddleOCR-VL output. It does not invent missing values or claim calibrated probabilities.

## 4. Checkpoint-free BPI rule classifier

No LayoutLMv3 model, `CLASSIFIER_MODEL_DIR`, model training, or external classifier checkpoint is required. The Python rule engine is defined in section 5 and records the applied rule and evidence for each page.

In [6]:
# Rule-based classifier: no LayoutLMv3, Transformers, Tesseract, or model checkpoint required.
import re
from collections import Counter
BPI_CLASSES = {"CLAIM_FORM", "INVOICE", "MEDICAL_RECORD", "ESTIMATE", "PHARMACY", "ONLINE_PROVIDER", "INCOMPLETE", "OTHER_UNKNOWN"}
print("BPI rule-based classification enabled; no classifier checkpoint is required.")

BPI rule-based classification enabled; no classifier checkpoint is required.


In [7]:
# Page ingestion is handled by PyMuPDF in section 3. No second PDF renderer or OCR fallback is used.

## BPI rule vocabulary

Classification is based on text actually present in native PDF text and PaddleOCR-VL output. Rules are precedence-based and return their matched rule plus evidence. The classifier does not infer missing facts. If evidence is weak or conflicting, it returns `OTHER_UNKNOWN` or marks the page for human review.

In [9]:
# The former model-dependent apply_company_rules() implementation is intentionally removed.
# See classify_bpi_text() in section 5 for the checkpoint-free implementation.

In [10]:
# No model loading or inference is required for BPI rule-based classification.
# PaddleOCR-VL supplies the primary visual text/layout evidence.

## 5. PaddleOCR-VL-1.5 + BPI rule-based classification

Each page is OCR-processed once with PaddleOCR-VL-1.5. The OCR engine is initialized explicitly with `device="gpu:0"`; the runtime preflight verifies a real Paddle GPU tensor operation before documents are processed. Classification uses native PDF text plus PaddleOCR-VL output. No Tesseract fallback is used.

Qwen3-4B remains optional and is disabled by default. OCR runs first and is not dependent on Qwen availability.


In [11]:
# PaddleOCR-VL primary extraction + BPI rule-based page classification.
# Running OCR here ensures classification uses the same primary OCR evidence as extraction.

def _collect_ocr_text(x):
    out=[]
    if isinstance(x,str):
        if x.strip(): out.append(x.strip())
    elif isinstance(x,dict):
        for k,v in x.items():
            if str(k).lower() in {"text","rec_text","content","ocr_text","markdown","label"} and isinstance(v,str):
                if v.strip(): out.append(v.strip())
            else: out.extend(_collect_ocr_text(v))
    elif isinstance(x,(list,tuple)):
        for v in x: out.extend(_collect_ocr_text(v))
    return out

def _has(pattern, text):
    return bool(re.search(pattern,text,re.I|re.S))

def classify_bpi_text(text, submitted_pages=1):
    """Return canonical BPI class, rule, evidence and review state from available page text."""
    t=re.sub(r'\s+',' ',text or '').strip()
    lower=t.lower()
    # Business rule requested: any page explicitly containing the word "invoice" is classified as INVOICE.
    if re.search(r'\binvoice\b', t, re.I):
        return {
            "predicted_class": "INVOICE", "model_class": "BPI_RULES",
            "final_class": "INVOICE", "display_label": "INVOICE",
            "confidence": 0.98, "processing_type": "INVOICE",
            "decision": "RULE_CLASSIFIED", "review_required": False,
            "review_reasons": [], "rule_applied": "R0_EXPLICIT_INVOICE_KEYWORD",
            "rule_reason": "The word invoice is explicitly present in the document text.",
            "evidence": {"invoice_keyword": True},
            "ocr_status": "TEXT_AVAILABLE" if t else "EMPTY",
            "ocr_words": len(t.split()), "classifier_ocr_text": t[:6000]
        }
    evidence={
      "claim": [p for p in [r'claim\s*form',r'claimant',r'policy\s*(?:no|number|#|holder)',r'insured',r'date\s+of\s+(?:loss|illness|injury|accident)'] if _has(p,t)],
      "billing": [p for p in [r'\binvoice\b',r'\binvoice\s*(?:no|number|#)',r'\bsubtotal\b',r'\btotal\s+(?:due|charges|amount)\b',r'\bqty\b|\bquantity\b',r'\bunit\s+price\b',r'\bamount\b|\bprice\b',r'\bbalance\s+due\b'] if _has(p,t)],
      "planned": [p for p in [r'\bestimate\b',r'treatment\s+plan',r'\bdraft\b',r'\bproposed\b',r'\bplanned\b',r'\bquote\b',r'\blow\b.{0,30}\bhigh\b',r'\bhigh\b.{0,30}\blow\b'] if _has(p,t)],
      "clinical": [p for p in [r'medical\s+(?:record|history)',r'\bdiagnos\w*',r'\bassessment\b',r'\bsubjective\b',r'\bobjective\b',r'physical\s+exam',r'\bvital signs\b',r'\btemperature\b',r'\bprognosis\b'] if _has(p,t)],
      "pharmacy": [p for p in [r'\bpharmacy\b',r'\bprescription\b',r'\bRx\s*(?:no|number|#)?\b',r'\bNDC\b',r'\bdispensed\b',r'\brefills?\b',r'\bpharmacist\b'] if _has(p,t)],
      "online": [p for p in [r'\border\s*(?:no|number|#|id)\b',r'\bshipping\b',r'\btracking\b',r'\border\s+summary\b',r'\bitems?\s+ordered\b',r'\bwww\.|https?://',r'\be-?receipt\b',r'\btrack\s+package\b',r'\bitems?\s+subtotal\b',r'\border\s+total\b',r'\border\s+placed\b',r'\bbuy\s+again\b'] if _has(p,t)],
      "payment": [p for p in [r'payment\s+(?:slip|receipt|received|method)',r'amount\s+tendered',r'authorization\s+(?:code|#)',r'\bvisa\b|\bmastercard\b|\bamex\b',r'thank\s+you\s+for\s+your\s+payment'] if _has(p,t)]
    }
    money_count=len(re.findall(r'(?<!\w)\$?\d[\d,]*\.\d{2}(?!\w)',t))
    page_refs=re.findall(r'\bpage\s*(\d+)\s*(?:of|/)\s*(\d+)\b',t,re.I)
    reasons=[]; rule='R8_UNRESOLVED'; final='OTHER_UNKNOWN'; confidence=0.45
    # R1: explicit missing pages or payment slip without any bill in the submitted file.
    missing=[(int(a),int(b)) for a,b in page_refs if int(b)>submitted_pages]
    if missing:
        final='INCOMPLETE'; rule='R1_MISSING_PAGES'; confidence=.99
        reasons.append(f"Document indicates page {missing[0][0]} of {missing[0][1]}, but only {submitted_pages} page(s) were submitted.")
    elif evidence['payment'] and not _has(r'\binvoice\b|\bsubtotal\b|\bqty\b|\bquantity\b|\btotal\s+(?:due|charges|amount)\b|\bbalance\s+due\b',t) and money_count <= 2:
        final='INCOMPLETE'; rule='R1_PAYMENT_SLIP_WITHOUT_INVOICE'; confidence=.88
        reasons.append('Payment-slip/payment evidence is present without sufficient invoice billing evidence.')
    # R2: claim form and invoice on the same page -> invoice.
    elif evidence['claim'] and evidence['billing'] and not evidence['pharmacy'] and not evidence['online']:
        final='INVOICE'; rule='R2_CLAIM_FORM_PLUS_INVOICE'; confidence=.94
        reasons.append('Claim-form and billing indicators occur together; BPI rule routes the page as an invoice.')
    # R3: pharmacy and online-provider documents.
    elif evidence['pharmacy'] and evidence['online']:
        final='OTHER_UNKNOWN'; rule='R3_CONFLICT_PHARMACY_ONLINE'; confidence=.40
        reasons.append('Both pharmacy and online-provider indicators are present; manual review is required.')
    elif evidence['pharmacy']:
        final='PHARMACY'; rule='R3_PHARMACY'; confidence=.91
        reasons.append('Prescription/pharmacy-specific indicators were found.')
    elif evidence['online']:
        final='ONLINE_PROVIDER'; rule='R3_ONLINE_PROVIDER'; confidence=.90
        reasons.append('Online order, shipping, tracking, or electronic receipt indicators were found.')
    # R4: estimate/planned treatment versus completed bill.
    elif evidence['planned'] and not (evidence['billing'] and money_count>=2 and not _has(r'\b(?:draft|estimate|treatment\s+plan|proposed|planned)\b',t)):
        final='ESTIMATE'; rule='R4_PLANNED_TREATMENT'; confidence=.90
        reasons.append('Estimate, draft, quote, proposed/planned treatment, or LOW/HIGH pricing evidence was found.')
    elif evidence['billing'] and money_count>=2:
        final='INVOICE'; rule='R4_COMPLETED_BILLING'; confidence=.88
        reasons.append('Completed billing indicators and multiple monetary values were found.')
    # R5: clinical records versus claim forms.
    elif evidence['clinical'] and not evidence['billing']:
        final='MEDICAL_RECORD'; rule='R5_CLINICAL_WITHOUT_BILLING'; confidence=.86
        reasons.append('Clinical/medical-record indicators are present without sufficient billing evidence.')
    elif evidence['claim'] and not evidence['billing']:
        final='CLAIM_FORM'; rule='R5_CLAIM_FORM_WITHOUT_BILLING'; confidence=.88
        reasons.append('Claim/policy/insured fields are present without invoice billing evidence.')
    elif evidence['clinical'] and evidence['billing'] and money_count>=2:
        final='INVOICE'; rule='R5_BILLING_DOCUMENT_WITH_CLINICAL_DETAILS'; confidence=.82
        reasons.append('Clinical content is accompanied by sufficient billing evidence; treated as the bill.')
    else:
        reasons.append('Available text does not provide enough reliable evidence for one BPI class.')
    review=(final in {'INCOMPLETE','OTHER_UNKNOWN'} or confidence<.80 or len(t.split())<8)
    if review: reasons.append('Human review required: class confidence/evidence is insufficient or the class is inherently exceptional.')
    return {'predicted_class':final,'model_class':'BPI_RULES','final_class':final,
            'display_label':'Online Store' if final=='ONLINE_PROVIDER' else final,
            'confidence':confidence,'processing_type':'INVOICE' if final in {'INVOICE','PHARMACY','ONLINE_PROVIDER'} else final,
            'decision':'HUMAN_REVIEW' if review else 'RULE_CLASSIFIED','review_required':review,
            'review_reasons':reasons if review else [],'rule_applied':rule,'rule_reason':' '.join(reasons),
            'evidence':evidence,'ocr_status':'TEXT_AVAILABLE' if t else 'EMPTY','ocr_words':len(t.split()),
            'classifier_ocr_text':t[:6000]}

# Initialize PaddleOCR-VL once, process every page once, and classify using native + Paddle text.
PADDLE_VL=None
def load_extractor():
    global PADDLE_VL
    if PADDLE_VL is None:
        try:
            import paddle
            from paddleocr import PaddleOCRVL
            if not paddle.is_compiled_with_cuda():
                raise RuntimeError("GPU OCR requested, but PaddlePaddle is CPU-only. Install paddlepaddle-gpu and restart runtime.")
            paddle.set_device("gpu:0")
            PADDLE_VL=PaddleOCRVL(device="gpu:0")
        except Exception as e:
            raise RuntimeError('PaddleOCR-VL initialization failed. No substitute OCR engine is used.') from e
    return PADDLE_VL

def jsonable(x):
    if x is None or isinstance(x,(str,int,float,bool)): return x
    if isinstance(x,dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [jsonable(v) for v in x]
    if hasattr(x,'json'):
        v=x.json
        return jsonable(v() if callable(v) else v)
    if hasattr(x,'to_dict'): return jsonable(x.to_dict())
    return str(x)

def _prepare_for_ocr(img):
    """Dark-mode screenshots (e.g. phone order pages) are inverted to dark-on-light before OCR. Returns (image, was_inverted)."""
    try:
        from PIL import ImageOps
        g=img.convert('L'); small=g.resize((max(1,g.width//8),max(1,g.height//8)))
        px=list(small.getdata())
        if sum(px)/max(1,len(px))<100:
            return ImageOps.invert(img.convert('RGB')), True
    except Exception:
        pass
    return img, False

def extract_page(page):
    engine=load_extractor(); tmp=ROOT/'_page.png'
    ocr_img,inverted=_prepare_for_ocr(page['image']); page['ocr_image_inverted']=inverted
    ocr_img.save(tmp)
    try: return jsonable(engine.predict(input=str(tmp)))
    finally: tmp.unlink(missing_ok=True)

page_runs=[]
for p in all_pages:
    started=time.perf_counter(); raw=extract_page(p); elapsed=round(time.perf_counter()-started,3)
    rec={k:v for k,v in p.items() if k!='image'}
    rec.update({'raw_output':raw,'elapsed_seconds':elapsed})
    page_runs.append(rec)
    safe_stem=re.sub(r'[^A-Za-z0-9_.-]+','_',Path(p['source_file']).stem)
    (RAW_DIR/f"{safe_stem}_p{p['page_number']}.json").write_text(json.dumps(rec,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# Release PaddleOCR-VL GPU allocations before optional Qwen inference.
# OCR has already completed for every page; this allows the 4B model to use the T4 afterwards.
def release_paddle_ocr_gpu():
    global PADDLE_VL
    import gc
    try:
        PADDLE_VL = None
        gc.collect()
        import paddle
        if paddle.is_compiled_with_cuda():
            paddle.device.cuda.empty_cache()
    except Exception as exc:
        print("GPU cache cleanup note:", repr(exc))

release_paddle_ocr_gpu()

classification_results=[]
for filename in sorted({p['source_file'] for p in all_pages}):
    file_pages=[p for p in all_pages if p['source_file']==filename]
    for page in file_pages:
        run=next(r for r in page_runs if r['source_file']==filename and r['page_number']==page['page_number'])
        text='\n'.join([page.get('native_text','')]+_collect_ocr_text(run.get('raw_output',[])))
        result=classify_bpi_text(text,submitted_pages=len(file_pages))
        result.update({'source_file':filename,'page_number':page['page_number'],'page_count':page.get('page_count',len(file_pages))})
        classification_results.append(result)

classification_df=pd.DataFrame([{k:v for k,v in r.items() if k!='evidence'} for r in classification_results])
display(classification_df[[c for c in ['source_file','page_number','final_class','display_label','confidence','processing_type','decision','review_required','rule_applied'] if c in classification_df.columns]])
classification_by_page={(r['source_file'],r['page_number']):r for r in classification_results}
print('PaddleOCR-VL pages:',len(page_runs),'| Rule-classified pages:',len(classification_results))

Creating model: ('PP-DocLayoutV3', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-DocLayoutV3), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-DocLayoutV3`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PaddleOCR-VL-1.6-0.9B', None, None)
Using official model (PaddleOCR-VL-1.6), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PaddleOCR-VL-1.6`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Bucketed engine_config has no entry for resolved engine 'paddle_dynamic'; using an empty config for that engine.
Loading configuration file /root/.paddlex/official_models/PaddleOCR-VL-1.6/config.json
Loading weights file /root/.paddlex/official_models/PaddleOCR-VL-1.6/model.safetensors
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_he

,source_file,page_number,final_class,display_label,confidence,processing_type,decision,review_required,rule_applied
0,23026778.pdf,1,INVOICE,INVOICE,0.98,INVOICE,RULE_CLASSIFIED,False,R0_EXPLICIT_INVOICE_KEYWORD


PaddleOCR-VL pages: 1 | Rule-classified pages: 1


## 6. PaddleOCR-VL raw OCR output

Raw OCR responses are preserved in `page_runs` and `outputs/raw_ocr/`.

## 6. PaddleOCR-VL extraction

PaddleOCR-VL was run once per page in section 5 so the same primary OCR/layout output supports classification and line-item extraction. `page_runs` contains the preserved raw responses. No Tesseract fallback is used.

## 7. Extract metadata for all document types and line items for eligible pages

Extract invoice number/date and patient names from explicit document evidence first. Then extract table rows using the Patient, Description, Quantity and Total columns when available. Payment-history fields such as Amount Paid and Payment Visa are excluded. Ambiguous rows are sent for review.


v4.6: pharmacy and online-order pages are parsed from text when PaddleOCR-VL returns no HTML table. Pet names are read from Pet Name / Patient Name / Patient / Animal / Prescribed For (Canine = dog, Feline = cat). A document that names exactly one pet assigns it to rows without their own pet label (recorded in `pet_assignment_source`); with several pets, unassigned rows stay `UNKNOWN` and are flagged.


**Metadata date rule:** The updated extractor recognizes month-name dates (for example, `Sept 22, 2026`), US numeric dates (for example, `10/22/2026`), day-first numeric dates when the first component is greater than 12 (for example, `22-10-2026`), and ISO dates. Per the requested rule, the chronologically latest date found across the same uploaded document is assigned to `invoice_date`. The full list is retained in `all_document_dates` for audit. If a document contains a future date, DOB, or unrelated date later than the invoice date, this rule can select the wrong date; such cases should be reviewed.


In [12]:
from html.parser import HTMLParser
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import re

def collect_text(x):
    """Collect readable OCR text without assuming one PaddleOCR response schema."""
    out = []
    if isinstance(x, str):
        if x.strip():
            out.append(x.strip())
    elif isinstance(x, dict):
        for k, v in x.items():
            if str(k).lower() in {"text", "rec_text", "content", "ocr_text", "label", "markdown"} and isinstance(v, str):
                if v.strip():
                    out.append(v.strip())
            else:
                out.extend(collect_text(v))
    elif isinstance(x, (list, tuple)):
        for v in x:
            out.extend(collect_text(v))
    return out

class _TableParser(HTMLParser):
    """Extract rows/cells from HTML tables embedded in PaddleOCR-VL output."""
    def __init__(self):
        super().__init__()
        self.rows, self.row, self.cell = [], None, None
    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag == "tr":
            self.row = []
        elif tag in ("td", "th") and self.row is not None:
            self.cell = []
    def handle_data(self, data):
        if self.cell is not None:
            self.cell.append(data)
    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag in ("td", "th") and self.cell is not None:
            self.row.append(" ".join("".join(self.cell).split()))
            self.cell = None
        elif tag == "tr" and self.row is not None:
            if self.row:
                self.rows.append(self.row)
            self.row = None

def html_tables(text):
    parser = _TableParser()
    try:
        parser.feed(text or "")
    except Exception:
        pass
    return parser.rows

MONEY_RE = re.compile(r'^\(?-?\$?\s*\d[\d,]*\.\d{2}\)?$')
DATE_RE = re.compile(r'\b(\d{1,2}/\d{1,2}/\d{4})\b')
INVOICE_RE = re.compile(r'\bInvoice\s*(?:Number|#|No\.?)?\s*[:#]?\s*([A-Z0-9-]+)', re.I)

# Robust date extraction. Ambiguous numeric dates are interpreted as US MM/DD/YYYY
# unless the first component is >12, in which case they are DD/MM/YYYY.
_DATE_TOKEN_RE = re.compile(
    r'(?i)\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
    r'Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\.?\s+\d{1,2}(?:st|nd|rd|th)?[,]?\s+\d{2,4}\b'
    r'|\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'
    r'|\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b'
)

def _parse_document_date(token):
    from datetime import datetime
    import calendar
    value = str(token).strip().replace(",", "")
    value = re.sub(r'\bSept\b', 'Sep', value, flags=re.I)
    value = re.sub(r'(\d)(st|nd|rd|th)\b', r'\1', value, flags=re.I)
    # ISO year-first
    if re.fullmatch(r'\d{4}[/-]\d{1,2}[/-]\d{1,2}', value):
        for fmt in ("%Y-%m-%d", "%Y/%m/%d"):
            try: return datetime.strptime(value, fmt).date()
            except ValueError: pass
    # Numeric dates: US month/day/year by default; if month > 12, use day/month/year.
    m = re.fullmatch(r'(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})', value)
    if m:
        a, b, y = map(int, m.groups())
        if y < 100: y += 2000 if y < 70 else 1900
        fmt = "%d/%m/%Y" if a > 12 else "%m/%d/%Y"
        try: return datetime.strptime(f"{a}/{b}/{y}", fmt).date()
        except ValueError: return None
    # Month-name dates
    for fmt in ("%B %d %Y", "%b %d %Y", "%d %B %Y", "%d %b %Y",
                "%B %d %y", "%b %d %y", "%d %B %y", "%d %b %y"):
        try: return datetime.strptime(value, fmt).date()
        except ValueError: pass
    return None

def extract_all_dates(text):
    found = []
    for token in _DATE_TOKEN_RE.findall(text or ""):
        d = _parse_document_date(token)
        if d:
            found.append((d, token))
    # De-duplicate by actual calendar date, preserving the first printed form.
    dedup = {}
    for d, token in found:
        dedup.setdefault(d, token)
    return sorted(dedup.items(), key=lambda x: x[0])

def dec(v):
    """Parse a decimal; tolerate trailing taxable marker T, e.g. '$15.76 T'."""
    s = str(v or "").strip().replace("$", "").replace(",", "").replace(" ", "")
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    s = re.sub(r"(?<=\d)[Tt]$", "", s)
    try:
        d = Decimal(s)
        return -d if neg else d
    except (InvalidOperation, ValueError):
        return None

def money(d):
    return None if d is None else str(d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

def extract_invoice_metadata(text):
    """Extract invoice number and latest valid document date from all common date formats."""
    raw = re.sub(r"<[^>]+>", " ", text or "")
    inv = INVOICE_RE.search(raw)
    dates = extract_all_dates(raw)
    latest = dates[-1] if dates else (None, None)
    return {
        "invoice_number": inv.group(1) if inv else None,
        "invoice_date": latest[1],
        "invoice_date_iso": latest[0].isoformat() if latest[0] else None,
        "all_document_dates": [token for _, token in dates],
        "patient_names": []
    }

def _find_col(header, names):
    normalized = [re.sub(r'[^a-z]', '', str(x).lower()) for x in header]
    for wanted in names:
        wanted = re.sub(r'[^a-z]', '', wanted.lower())
        for i, h in enumerate(normalized):
            if h == wanted:
                return i
    return None

def parse_html_invoice_tables(raw, filename, page, method):
    """Parse genuine invoice rows; retain decimal quantities and reject courtesy/sign-off rows."""
    found = []
    non_item_patterns = re.compile(
        r'^(?:thank\s+you\b|thanks\b|subtotal\b|invoice\s+total\b|grand\s+total\b|'
        r'total\b|amount\s+paid\b|paid\b|payment\b|balance\b|amount\s+remaining\b|'
        r'change\s+due\b|cash\s+tendered\b|visa\b|mastercard\b|signature\b)', re.I
    )
    tax_label = re.compile(r'^(?:tax|sales\s+tax|state\s+tax|local\s+tax|gst|hst|vat)\b', re.I)

    for blob in collect_text(raw):
        clean_blob = re.sub(r"<[^>]+>", " ", blob)
        heading_match = re.search(
            r"\b(?:patient|pet|animal|prescribed\s+for)\s*(?:name)?\s*[:#-]\s*([A-Z][A-Za-z' -]{1,40})",
            clean_blob, re.I
        )
        table_pet_heading = heading_match.group(1).strip() if heading_match else None
        table = html_tables(blob)
        if not table:
            continue

        header_idx = next(
            (i for i, row in enumerate(table)
             if any(re.search(r'\b(description|item|service|product|medication)\b', str(c), re.I) for c in row)
             and any(re.search(r'\b(quantity|qty|amount|total|extended|price|cost)\b', str(c), re.I) for c in row)),
            None
        )
        if header_idx is None:
            continue

        header = table[header_idx]
        c_pet = _find_col(header, ["Patient", "Pet", "Animal"])
        c_desc = _find_col(header, ["Description", "Item", "Service", "Product", "Medication", "Procedure", "Details"])
        c_qty = _find_col(header, ["Quantity", "Qty"])
        c_total = _find_col(header, ["Line Total", "Extended Price", "Extended Amount", "Amount", "Total", "Price", "Cost"])
        c_sub = _find_col(header, ["Subtotal", "Sub Total"])
        c_tax = _find_col(header, ["Tax"])
        c_date = _find_col(header, ["Date", "Service Date"])
        if c_desc is None:
            continue

        for row in table[header_idx + 1:]:
            if c_desc >= len(row):
                continue
            desc = str(row[c_desc]).strip()
            if not desc or non_item_patterns.search(desc):
                continue

            amount_idx = c_total if c_total is not None and c_total < len(row) else c_sub
            if amount_idx is None or amount_idx >= len(row):
                continue
            amount = dec(row[amount_idx])
            if amount is None:
                continue

            pet = row[c_pet].strip() if c_pet is not None and c_pet < len(row) else table_pet_heading

            # An explicit tax row is included only when a non-zero tax amount is actually printed.
            if tax_label.match(desc):
                if amount != 0:
                    found.append({
                        "description": "Tax", "quantity": 1, "allowed_amount": money(amount),
                        "pet_name": pet or None, "service_date": None, "row_tax": None,
                        "line_type": "tax", "source_file": filename, "page_number": page,
                        "source_text": " | ".join(str(x) for x in row),
                        "evidence_line_number": None, "extraction_method": method,
                        "review_required": False, "review_reasons": []
                    })
                continue

            qty_value = dec(row[c_qty]) if c_qty is not None and c_qty < len(row) else None
            # Missing quantity stays missing; never invent quantity=1.
            qty = format(qty_value, "f") if qty_value is not None else None
            service_date = row[c_date].strip() if c_date is not None and c_date < len(row) else None
            # Only numeric values in an explicit tax column count as tax; 'T' is just a marker.
            row_tax = dec(row[c_tax]) if c_tax is not None and c_tax < len(row) else None

            found.append({
                "description": desc, "quantity": qty, "allowed_amount": money(amount),
                "pet_name": pet or None, "service_date": service_date or None,
                "row_tax": row_tax, "line_type": "item", "source_file": filename,
                "page_number": page, "source_text": " | ".join(str(x) for x in row),
                "evidence_line_number": None, "extraction_method": method,
                "review_required": qty is None,
                "review_reasons": ["quantity_not_printed_or_uncertain"] if qty is None else []
            })
    return found


def parse_rows(text, filename, page, method):
    """Conservative plain-text fallback. HTML table rows are handled separately."""
    items, taxes, summaries = [], [], []
    for ln, raw in enumerate((text or "").splitlines(), 1):
        line = " ".join(raw.split()).strip(" |")
        if not line or "<tr" in line.lower() or "<td" in line.lower():
            continue
        if re.search(r'\b(amount paid|payment visa|paid in transaction|paid to date|amount remaining)\b', line, re.I):
            continue
        matches = list(re.finditer(r'\$?\s*\(?-?\d[\d,]*\.\d{2}\)?', line))
        if not matches:
            continue
        label = line
        if re.match(r'^(tax|sales tax|gst|hst|vat)\b', label, re.I):
            amount = dec(matches[-1].group())
            if amount is not None and amount != 0:
                taxes.append({"description":"Tax","quantity":1,"allowed_amount":money(amount),
                    "line_type":"tax","source_file":filename,"page_number":page,
                    "source_text":raw,"evidence_line_number":ln,"extraction_method":method,
                    "review_required":False,"review_reasons":[],"pet_name":None,"service_date":None})
            continue
        if re.match(r'^(subtotal|invoice total|grand total|total|discount|coupon|credit|refund)\b', label, re.I):
            summaries.append({"label":label,"amount":money(dec(matches[-1].group())),
                "source_file":filename,"page_number":page,"source_text":raw,"line_number":ln})
            continue
        # Do not guess columns from arbitrary free text; leave for manual review.
    return items, taxes, summaries


## 7b. Text-based line-item parsers (pharmacy invoices, online-store order pages, screenshots)

Used only when PaddleOCR-VL returns no HTML table for a billing page. Prices are aligned to the printed price columns; a plain integer left of the first price is the quantity; totals, payments and thank-you lines are never rows.

In [13]:
# ===== Section 7b: line items from pharmacy invoices, online-order screenshots and plain-text bills =====
# Deterministic Python only. Used when a billing page (INVOICE / PHARMACY / ONLINE_PROVIDER) has no HTML table that parse_html_invoice_tables() could read.
# Rules applied (from the ORCA understanding rules): never pick a number just because it is the last one; use header / label relationships;
# keep tax, shipping, discounts, payments and totals out of the medical line items; keep zero-dollar items and repeated rows; flag anything ambiguous.
MONEY_TOK_RE = re.compile(r'\(?-?\$?\s*\d[\d,]*\.\d{2}\)?')
SUMMARY_LINE_RE = re.compile(
    r'^\W*(?:thank\s*you|thanks|(?:items?\s+|patient\s+)?sub\s*-?\s*total|(?:invoice|order|grand)\s+total|total\b|amount\s+paid|paid\b|payment|'
    r'balance|amount\s+remaining|change\s+due|cash\s+tendered|visa\b|mastercard\b|signature|shipping|flat-?\s*rate|delivery\s+fee|'
    r'sales\s+tax|tax\b|discount|coupon|promo|credit\b|refund|savings)', re.I)
BOILERPLATE_RE = re.compile(r'\b(?:add\s+to\s+cart|track\s+package|delivered|placed|buy\s+again|view\s+order|reorder|write\s+a\s+review|return\s+or\s+replace)\b', re.I)
QTY_LINE_RE = re.compile(r'^\W{0,3}Qty\s*[:.]?\s*(\d+(?:\.\d+)?)\b(.*)$', re.I)
STAFF_TAIL_RE = re.compile(r"\s+[A-Z][a-z]+(?:\s+[A-Z]\.?)?\s+[A-Z][A-Za-z'-]+,?\s*(?:DVM|VMD|DACVS|LVT|CVT|RVT|VT|VA|CVPM)\b.*$")
# header keywords, evaluated left to right in ONE pass so 'Unit Price' is not also read as 'Price' / 'Total'
_HDR_RE = re.compile(
    r'(?P<date>\bdate\b|\bdos\b)|(?P<description>\bdescription\b|\bitem\b|\bmedication\b|\bproduct\b|\bdrug\b|\bservice\b|\bprocedure\b|\bdetails\b)|'
    r'(?P<staff>\bstaff(?:\s+member)?\b|\bprovider\b|\bdoctor\b|\bperformed\s+by\b)|(?P<quantity>\bqty\b|\bquantity\b|\bunits?\b)|'
    r'(?P<unit_price>\bunit\s+(?:price|cost)\b|\bprice\s+each\b|\beach\b|\brate\b)|'
    r'(?P<line_total>\bextended(?:\s+(?:price|amount))?\b|\bext\.?\s*(?:price|amt|amount)?\b|\bline\s+total\b|\bline\s+amount\b|\bamount\b|\btotal\b|\bcharge\b|\bcharges\b)|'
    r'(?P<price>\bprice\b)|(?P<tax>\btax\b)|(?P<discount>\bdiscount\b)', re.I)


def _norm_lines(text):
    out = []
    for raw in (text or '').splitlines():
        s = ' '.join(raw.replace(' ', ' ').split()).strip(' |')
        if s:
            out.append(s)
    return out


def _header_kinds(line):
    """Ordered column kinds if `line` is a table header (needs a description column and a quantity or price/amount column), else None."""
    kinds = []
    for m in _HDR_RE.finditer(line):
        k = m.lastgroup
        if k == 'price':
            k = 'unit_price' if ('quantity' in kinds and 'line_total' not in kinds and 'unit_price' not in kinds) else 'line_total'
        if k not in kinds:
            kinds.append(k)
    if 'description' in kinds and ({'quantity', 'unit_price', 'line_total'} & set(kinds)) and len(line.split()) <= 14 and not MONEY_TOK_RE.search(line):
        return kinds
    return None


def _row_dict(desc, qty, amount, filename, page, method, src, ln, flags, service_date=None, row_tax=None):
    return {"description": desc, "quantity": qty, "allowed_amount": money(amount), "pet_name": None, "service_date": service_date, "row_tax": row_tax,
            "line_type": "item", "source_file": filename, "page_number": page, "source_text": src, "evidence_line_number": ln,
            "extraction_method": method, "review_required": bool(flags), "review_reasons": list(flags)}


def parse_header_aware_text_rows(lines, filename, page, method):
    """Rows printed as one text line under a header such as 'Date Description Staff Member Qty Total' or 'Medication Qty Unit Price Amount'.
    Money tokens (with cents) are assigned to the printed price columns from the right; a plain integer immediately left of the first money token is the quantity.
    Anything that does not fit the header is flagged instead of guessed."""
    items, unattached, kinds = [], [], None
    for ln, s in enumerate(lines, 1):
        hk = _header_kinds(s)
        if hk and kinds is None:
            kinds = hk
            continue
        if kinds is None:
            continue
        if SUMMARY_LINE_RE.match(s):
            break
        toks = s.split()
        money_idx = [i for i, t in enumerate(toks) if re.fullmatch(r'\(?-?\$?\d[\d,]*\.\d{2}\)?[Tt]?', t)]
        if not money_idx:
            if items and (s[:1].islower() or s[:1] == '(') and not BOILERPLATE_RE.search(s):
                items[-1]["description"] += ' ' + s
                items[-1]["source_text"] += ' / ' + s
                items[-1]["review_reasons"] = list(dict.fromkeys(items[-1]["review_reasons"] + ['wrapped_description_merged']))
                items[-1]["review_required"] = True
            else:
                unattached.append(s)
            continue
        money_kinds = [k for k in kinds if k in ('unit_price', 'line_total', 'tax', 'discount')]
        flags = []
        first_money = money_idx[0]
        # right-align the money tokens to the printed money columns
        tail = money_idx[-len(money_kinds):] if money_kinds else money_idx[-1:]
        assign = dict(zip(money_kinds[-len(tail):], tail)) if money_kinds else {'line_total': tail[-1]}
        if len(money_idx) != len(money_kinds):
            flags.append('money_columns_do_not_match_header')
        if 'line_total' not in assign:
            flags.append('line_total_column_missing')
            assign['line_total'] = money_idx[-1]
        amount = dec(toks[assign['line_total']])
        unit = dec(toks[assign['unit_price']]) if 'unit_price' in assign else None
        qty, body_end = None, min(money_idx)
        if 'quantity' in kinds and body_end > 0 and re.fullmatch(r'\d+(?:\.\d+)?', toks[body_end - 1]):
            qty = toks[body_end - 1]
            body_end -= 1
        else:
            flags.append('quantity_uncertain')
        start = 0
        service_date = None
        if 'date' in kinds and toks and re.fullmatch(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', toks[0]):
            service_date, start = toks[0], 1
        desc = ' '.join(toks[start:body_end])
        if 'staff' in kinds:
            stripped = STAFF_TAIL_RE.sub('', desc).strip()
            if stripped != desc:
                desc = stripped
            else:
                flags.append('staff_member_not_separated_from_description')
        if len(re.findall(r'[A-Za-z]', desc)) < 3 or amount is None:
            unattached.append(s)
            continue
        if qty is not None:
            qd = dec(qty)
            if qd is not None and (qd >= 100 or qd <= 0):
                flags.append('quantity_unusual_check_against_description')
            if unit is not None and amount is not None and qd is not None and abs(qd * unit - amount) > Decimal('0.01'):
                flags.append('qty_x_unit_price_differs_from_printed_amount')
        row_tax = dec(toks[assign['tax']]) if 'tax' in assign else None
        items.append(_row_dict(desc, qty, amount, filename, page, method + '_text_header', s, ln, flags, service_date, row_tax))
    return items, unattached


def _clean_desc_line(s):
    prev = None
    while prev != s:
        prev = s
        s = re.sub(r'^(?:[^\w$(#]+|[A-Za-z]{1,2}[—–\-_=|~]+\s*|[a-z]\s+(?=[A-Z]))', '', s)
    s = re.sub(r'\s*(?:[¢©®°]\)?|\(?[A-Za-z]\))\s*$', '', s)
    return s.strip()


def parse_order_card_items(lines, filename, page, method):
    """Online-store order screens (e.g. 'Qty: 1  $99.99' under a product title). The product title is the text between the previous item/boilerplate and the Qty line;
    the price printed with the Qty is the extended amount. Truncated titles ('...') and Qty > 1 (unit vs extended price unclear) are flagged."""
    items, buf, i = [], [], 0
    while i < len(lines):
        s = lines[i]
        m = QTY_LINE_RE.match(s)
        if m:
            qty, rest = m.group(1), m.group(2)
            price, j = None, i
            mm = MONEY_TOK_RE.findall(rest)
            if mm:
                price = mm[-1]
            else:
                for k in range(i + 1, min(i + 3, len(lines))):
                    if MONEY_TOK_RE.fullmatch(lines[k].strip()):
                        price, j = lines[k].strip(), k
                        break
                    if QTY_LINE_RE.match(lines[k]):
                        break
            parts = [_clean_desc_line(x) for x in buf]
            desc = ' '.join(x for x in parts if len(re.findall(r'[A-Za-z]', x)) >= 3).strip()
            raw_src = ' / '.join(buf + [s])
            buf = []
            if desc and price and dec(price) is not None:
                flags = []
                if desc.endswith('...') or desc.endswith('…'):
                    flags.append('description_truncated_in_source')
                if dec(qty) != 1:
                    flags.append('price_basis_unit_or_extended_unclear')
                items.append(_row_dict(desc, qty, dec(price), filename, page, method + '_order_card', raw_src, i + 1, flags))
            i = j + 1
            continue
        if SUMMARY_LINE_RE.match(s) or BOILERPLATE_RE.search(s) or MONEY_TOK_RE.fullmatch(s):
            buf = []
        else:
            buf = (buf + [s])[-5:]
        i += 1
    return items


def extract_text_taxes(lines, filename, page, method):
    """Explicit, non-zero tax lines such as 'Sales Tax: $2.56' -> a separate Tax item (never inferred)."""
    taxes = []
    for ln, s in enumerate(lines, 1):
        m = re.match(r'^(?:sales\s+)?tax(?:es)?\b[^\d$]{0,25}\$?\s*(\d[\d,]*\.\d{2})\s*$', s, re.I)
        if m and dec(m.group(1)) not in (None, Decimal('0')):
            taxes.append({"description": "Tax", "quantity": 1, "allowed_amount": money(dec(m.group(1))), "pet_name": None, "service_date": None, "line_type": "tax",
                          "source_file": filename, "page_number": page, "source_text": s, "evidence_line_number": ln, "extraction_method": method + "_text_tax",
                          "review_required": False, "review_reasons": []})
    return taxes


def extract_adjustments(text, filename):
    """Shipping and discounts are audit metadata only - never line items. 'FREE' shipping is recorded as 0.00."""
    out = []
    flat = ' '.join((text or '').split())
    for m in re.finditer(r'\b(Flat-?\s*Rate\s+Shipping|Shipping(?:\s*&\s*Handling)?|Shipping\s+Fee|Delivery\s+Fee)\s*[:$ ]*\s*(FREE|\$?\s*[0-9,]+\.\d{2})', flat, re.I):
        amt = '0.00' if m.group(2).upper() == 'FREE' else money(dec(m.group(2)))
        out.append({"source_file": filename, "label": re.sub(r'\s+', ' ', m.group(1)).strip(), "kind": "shipping", "amount": amt, "source_text": m.group(0)})
    for m in re.finditer(r'\b(Discount|Coupon|Promo(?:tion)?(?:\s+Code)?|Savings)\b[^$\d(-]{0,20}(-?\$?\s*\(?[0-9,]+\.\d{2}\)?)', flat, re.I):
        out.append({"source_file": filename, "label": m.group(1), "kind": "discount", "amount": money(dec(m.group(2))), "source_text": m.group(0)})
    return out


def extract_rows_from_page_text(page_rec, run, filename, method):
    """Try PaddleOCR-VL text first, then the PDF text layer (never both for the same page, so rows are not counted twice)."""
    blobs = []
    for b in collect_text(run.get("raw_output", [])):
        if '<table' in b.lower() or '<tr' in b.lower() or b in blobs:
            continue
        blobs.append(b)
    sources = [("paddleocr_vl15_text", '\n'.join(blobs)), ("pymupdf_native_text", page_rec.get("native_text", ""))]
    for name, txt in sources:
        lines = _norm_lines(txt)
        if not lines:
            continue
        items, unattached = parse_header_aware_text_rows(lines, filename, page_rec["page_number"], name)
        if not items:
            items = parse_order_card_items(lines, filename, page_rec["page_number"], name)
            unattached = []
        taxes = extract_text_taxes(lines, filename, page_rec["page_number"], name)
        if items or taxes:
            return items, taxes, {"source": name, "rows": len(items), "taxes": len(taxes), "unattached_lines": unattached[:20]}
    return [], [], {"source": None, "rows": 0, "taxes": 0, "unattached_lines": []}


# ---- pet names: inline labels ("Animal: Bella", "Prescribed For: Max") and label-block layouts (labels listed first, values below) ----
PET_LABELS = r'(?:patient\s*name|pet\s*name|patient|pet|animal|prescribed\s+for)'
_NOT_A_PET = {'species', 'breed', 'age', 'sex', 'gender', 'name', 'id', 'canine', 'feline', 'dog', 'cat', 'invoice', 'date', 'total', 'subtotal', 'payment',
              'balance', 'customer', 'clinical', 'order', 'animal', 'staff', 'member', 'qty', 'description', 'credit', 'debit', 'method', 'amount', 'paid',
              'unknown', 'none', 'n/a', 'information', 'details', 'owner', 'client', 'weight', 'color', 'dob', 'microchip', 'number', 'rx', 'prescription'}


def _pet_ok(v):
    v = (v or '').strip(" ,.;:-'’")
    return bool(v) and v.lower() not in _NOT_A_PET and bool(re.fullmatch(r"[A-Z][A-Za-z'’-]{1,30}(?: [A-Z][A-Za-z'’-]{1,20})?|[A-Z]{3,20}", v, re.I))


def extract_pet_names(text):
    """
    Extract pet names from explicit pet context:
    Pet/Patient/Animal/Prescribed For labels, '# Name' lines, 'Ordered for Name',
    and name followed by a species marker such as 'Journey (dog)'.
    Species words themselves are never returned as names.
    """
    names, evidence = [], []
    lines = _norm_lines(text)
    seen = set()

    def add(candidate, source_line):
        candidate = re.sub(r'\s*\((?:dog|cat|canine|feline)\)\s*$', '', str(candidate or '').strip(), flags=re.I)
        candidate = candidate.strip(" ,.;:-#'’")
        if not candidate or not _pet_ok(candidate):
            return
        key = candidate.casefold()
        if key not in seen:
            seen.add(key)
            names.append(candidate)
            evidence.append(source_line)

    # Explicit labels, including # used as a separator: "Pet # Tilly".
    for ln in lines:
        patterns = [
            r'\b' + PET_LABELS + r'\s*(?:name)?\s*[:#-]\s*([A-Za-z][A-Za-z\'’-]*(?:\s+[A-Z][A-Za-z\'’-]*)?)',
            r'\bordered\s+for\s*[:#-]?\s*([A-Za-z][A-Za-z\'’-]*(?:\s+[A-Z][A-Za-z\'’-]*)?)',
            r'^\s*#\s*([A-Za-z][A-Za-z\'’-]*(?:\s+[A-Z][A-Za-z\'’-]*)?)(?=\s*(?:\||,|;|\(|$))',
            r'\b([A-Z][A-Za-z\'’-]{1,30})\s*\((?:dog|cat|canine|feline)\)',
        ]
        for pattern in patterns:
            for m in re.finditer(pattern, ln, re.I):
                add(m.group(1), ln)

    # Label-block layout: label line then one unique name before the next table/header.
    for i, ln in enumerate(lines):
        if re.fullmatch(PET_LABELS + r'\s*:?', ln, re.I):
            candidates = []
            for nxt in lines[i + 1:i + 14]:
                if re.search(r'\b(?:description|qty|quantity)\b', nxt, re.I):
                    break
                if re.fullmatch(r"[A-Za-z][A-Za-z'’-]{1,30}", nxt) and _pet_ok(nxt):
                    candidates.append(nxt)
            if len(set(candidates)) == 1:
                add(candidates[0], ln + " ... " + candidates[0])

    return names, evidence


## 8. BPI-specific rules and exact output schema

- Extract invoice number, invoice date and patient names as document metadata before preparing line items.
- For multi-pet invoices, retain the patient name on each line item from the Patient column.
- The compact Orca payload remains exactly `description`, `quantity`, `allowed`.
- Procedure dropdown/code is excluded; the operator selects it only when certain.
- Incident, Original, Discount, Submitted, Paid and Actions are excluded.
- Banfield: verify the Your Fee column; flag uncertain amounts.
- VCA: starred items require verification against the Care Club exclusion rule.
- Tax is a separate line item only when explicitly present and non-zero.
- Payment history, Amount Paid, Payment Visa, Paid to Date and Balance Remaining are not medical line items.
- Do not guess missing pet assignments or quantities.
- Qwen receives the ORCA document-understanding rules (next cell) as its system prompt. Python does all arithmetic; Qwen only interprets and must quote its evidence verbatim.
- Shipping and discounts stay out of the line items (`invoice_adjustments_audit.csv`); a printed order total is compared with items + explicit tax + shipping.


In [14]:
# ===== Section 8b: ORCA document-understanding rules for Qwen (system prompt) + Python validation of anything Qwen proposes =====
# The text below is the supplied rule set, unchanged. It is sent as the SYSTEM prompt on every Qwen call (metadata fallback and line-item fallback).
# Qwen only interprets; Python keeps the arithmetic, accepts a proposal only if its evidence is a verbatim quote of the OCR text, and never overwrites a value
# that deterministic extraction already found. Accepted Qwen rows always stay review_required.
QWEN_RULES = """# ORCA Veterinary Insurance Document Understanding Rules

You are a veterinary insurance document understanding assistant.

Your responsibility is to interpret OCR-extracted veterinary documents using contextual understanding, document structure and visual-layout evidence.

You support veterinary hospitals, pharmacies, online veterinary providers, receipts, prescriptions, medical records, estimates and insurance claim forms.

## 1. General document understanding

Understand the entire supplied document context before extracting individual fields.

Identify the document type using its actual content and structure.

Recognize that a document may contain multiple pages, multiple sections, multiple pets and multiple invoice-like blocks.

Do not assume that every document originates from a veterinary hospital.

Do not split one supplied invoice into multiple invoices merely because it contains multiple sections or number blocks.

## 2. Pet identification

Identify pet names from all available contextual labels, including:

- Pet Name
- Patient Name
- Patient
- Animal
- Prescribed For
- Prescription details
- Patient information sections
- Pet headings associated with medication or treatment rows

Understand that:
- Canine means dog.
- Feline means cat.
- Dog and Cat may appear in parentheses or beside patient information.

If Patient Name and Pet Name are absent, inspect Animal and other contextual labels.

Distinguish pet names from customer names, clinic names and veterinarian names.

For multiple pets, associate treatments with the correct pet only when supported by explicit document evidence.

If the association is uncertain, return UNKNOWN and require human review.

Never infer a pet assignment from a medication or treatment name.

## 3. Clinic and customer identification

Identify the clinic or provider independently from the customer.

Understand that:
- Provider names commonly appear in headers, logos and letterheads.
- Provider addresses may appear directly below provider names.
- Client addresses may appear in Bill To, Customer or Owner sections.
- Invoice numbers and dates may appear in separate columns or visually aligned fields.
- Pharmacy names and prescribing clinic names may be different.

Never interchange clinic and client addresses.

Use layout relationships and nearby labels, not just text proximity.

## 4. Invoice line items

Extract every genuine purchased veterinary service, medication or product.

Preserve the original description, including:
- Medicine name
- Strength
- Dosage form
- Pack size
- Product variant
- Procedure details

Do not omit unfamiliar products.

Retain genuine zero-dollar medical items and repeated rows.

Exclude:
- Thank-you messages
- Staff signatures
- Payment confirmations
- Account balances
- Subtotals
- Totals
- Customer notes
- Non-medical administrative messages

Do not classify a genuine medical item as non-medical merely because its amount is zero.

## 5. Numeric interpretation

Understand the relationship between description, quantity, unit price and extended amount.

Use the original table's row and column relationships.

Never select a number merely because it is the last number in an OCR row.

Do not confuse:
- Quantity with price
- Unit price with extended total
- Tax with item price
- Payment amount with invoice amount
- Invoice number with a monetary amount

Pay special attention to visually similar digits, including 6 and 8, 3 and 8, 0 and 6, and 1 and 7.

If OCR evidence is ambiguous, do not guess. Request targeted image verification and flag the value for review.

Never alter an amount to force reconciliation.

## 6. Pharmacy and online-provider documents

Recognize completed pharmacy purchases and online veterinary medication orders.

Extract medication names, strengths, quantities, prices, prescription information, pet names and customer information.

Understand the distinction between:
- Unit price
- Pack price
- Extended line total
- Discount
- Tax
- Shipping

Extract purchased medicines as line items.

Do not extract prescription instructions as purchased products unless an actual purchase line is present.

Do not treat shipping or discounts as medicines.

## 7. Tax and additional charges

Create a separate Tax item only if tax is explicitly shown and is not already included in medical line totals.

Use:
description = Tax
quantity = 1
allowed = explicitly printed tax amount

If tax is absent, do not create a tax item.

Keep shipping, discounts and payment information separate from medical line items.

## 8. Validation and uncertainty

Review each extracted description, quantity and amount against the supplied OCR evidence.

Identify:
- Missing line items
- Suspicious descriptions
- Missing quantities
- Suspicious prices
- Incorrect row associations
- Uncertain pet assignments
- Possible tax confusion
- Conflicting metadata
- Incomplete documents

If information is unavailable, return null.

If information is ambiguous, return an uncertainty flag and explain why.

Do not invent missing information.

Do not silently correct OCR values.

Python performs deterministic calculations and reconciliation. Your role is contextual interpretation, identifying ambiguity and recommending targeted verification.

## 9. Evidence requirements

For every proposed correction or extracted metadata value, provide the exact supporting OCR evidence or a short verbatim evidence quotation.

If the supplied evidence does not support the value, do not accept it as confirmed.

Clearly distinguish:
- Confirmed
- Uncertain
- Missing
- Requires image verification

## 10. Output requirements

Return only valid JSON matching the requested output schema.

Do not add explanatory prose outside the JSON.

Do not add fields to the Orca payload schema.

Do not fabricate values to complete the output.
"""

QWEN_SYSTEM_PROMPT = QWEN_RULES + """

## 11. Answer format for this call
Return ONLY one JSON object in the structure requested in the user message. Unknown pet -> the string "UNKNOWN". Missing value -> null.
Pet-name evidence may also be a line beginning with "# Name", "Ordered for Name", or a name followed by a species marker such as "Journey (dog)". Do not return the species word as the pet name.
Every returned value needs an exact verbatim quotation from the OCR text as evidence; without evidence the value will be discarded by Python."""

_SPECIES_WORDS = {"canine", "feline", "dog", "cat", "unknown", "none", "n/a"}


def validate_qwen_line_items(result, source_text, known_pets=()):
    """Accept a Qwen-proposed line item only if (1) its evidence quote occurs verbatim in the OCR text, (2) every description word occurs in the OCR text,
    (3) the amount is a money value printed in the OCR text, (4) a quantity appears in the evidence, (5) it is not a total / tax / shipping / payment row.
    A pet is kept only if it is a known pet of the document; otherwise it becomes UNKNOWN."""
    accepted, rejected = [], []
    src = _normalise_evidence(source_text)
    src_money = {dec(m) for m in MONEY_TOK_RE.findall(source_text or "")} - {None}
    pets = {str(p).strip().casefold(): str(p).strip() for p in known_pets if p}
    items = result.get("line_items") if isinstance(result, dict) else None
    if not isinstance(items, list):
        return [], [{"item": None, "reasons": ["no_line_items_list_in_qwen_output"]}]
    for it in items:
        reasons = []
        if not isinstance(it, dict):
            rejected.append({"item": it, "reasons": ["not_an_object"]})
            continue
        desc = str(it.get("description") or "").strip()
        ev = str(it.get("evidence") or "").strip()
        amount = dec(it.get("allowed"))
        if not desc:
            reasons.append("empty_description")
        if not ev or _normalise_evidence(ev) not in src:
            reasons.append("evidence_not_verbatim_in_ocr_text")
        words = [w for w in re.findall(r"[A-Za-z0-9]{3,}", desc)]
        if desc and not all(w.casefold() in src for w in words):
            reasons.append("description_words_not_in_ocr_text")
        if amount is None or amount not in src_money:
            reasons.append("amount_not_printed_in_ocr_text")
        qty = it.get("quantity")
        qty_s = None if qty in (None, "") else str(qty).strip()
        if qty_s is not None and qty_s not in re.findall(r"\d+(?:\.\d+)?", ev):
            reasons.append("quantity_not_in_evidence")
        if desc and SUMMARY_LINE_RE.match(desc):
            reasons.append("summary_tax_shipping_or_payment_row")
        if reasons:
            rejected.append({"item": it, "reasons": reasons})
            continue
        pet = str(it.get("pet_name") or "").strip()
        pet = pets.get(pet.casefold()) if pet and pet.upper() != "UNKNOWN" else None
        accepted.append({"description": desc, "quantity": qty_s, "allowed_amount": money(amount), "pet_name": pet, "service_date": None, "row_tax": None,
                         "line_type": "item", "evidence_quote": ev,
                         "review_required": True,
                         "review_reasons": ["qwen_fallback_row_requires_human_review"] + (["quantity_uncertain"] if qty_s is None else [])})
    return accepted, rejected


def validate_qwen_pet_names(value, evidence_quote, source_text, client_name=None):
    """Pet names proposed by Qwen are accepted one by one: name must be inside the verbatim evidence quote, the quote must be in the OCR text,
    the name must not be a species word and must not be the client's name."""
    names = value if isinstance(value, list) else ([value] if isinstance(value, str) else [])
    quote = _normalise_evidence(evidence_quote)
    src = _normalise_evidence(source_text)
    ok = []
    for n in names:
        n = str(n or "").strip()
        if not n or n.upper() == "UNKNOWN" or n.casefold() in _SPECIES_WORDS:
            continue
        if not quote or quote not in src or n.casefold() not in quote:
            continue
        if client_name and n.casefold() in str(client_name).casefold():
            continue
        if n not in ok:
            ok.append(n)
    return ok


In [21]:
rows = []
tax_rows = []
invoice_summaries = []
invoice_adjustments = []   # shipping / discounts: audit metadata only, never line items
page_extraction_log = []
all_document_metadata = []

# Metadata extraction is performed for every classified document type.
# Line-item extraction is gated by class, with a later table-detection override.

def clean_ocr_text(text):
    return " ".join(re.sub(r"<[^>]+>", " ", text or "").split())

def first_match(patterns, text, flags=re.I):
    for pattern in patterns:
        m=re.search(pattern,text,flags)
        if m: return m.group(1).strip(" ,;:-")
    return None


def extract_clinic_header_metadata(text):
    """Extract clinic letterhead from the top of the page, keeping clinic and client blocks separate."""
    lines = [re.sub(r'\s+', ' ', x).strip(" \t|,:;")
             for x in re.sub(r'<[^>]+>', '\n', text or '').splitlines()]
    lines = [x for x in lines if x]
    header = lines[:30]
    clinic_name = None
    clinic_idx = None
    clinic_re = re.compile(
        r'\b(?:hospital|veterinary|veterinarian|vet\s+clinic|animal\s+clinic|pet\s+hospital)\b',
        re.I
    )
    excluded = re.compile(r'\b(?:bill\s+to|ship\s+to|client|customer|owner|patient|pet\s+name)\b', re.I)

    for i, line in enumerate(header):
        if clinic_re.search(line) and not excluded.search(line):
            # Avoid retaining phone/email text attached to the letterhead.
            candidate = re.split(r'\b(?:phone|ph:|fax|email:|www\.)\b', line, maxsplit=1, flags=re.I)[0].strip(" ,;:-")
            if len(candidate) >= 4:
                clinic_name, clinic_idx = candidate, i
                break

    # Address lines immediately below the clinic heading are the clinic's address.
    street_re = re.compile(
        r'^\d{1,6}\s+[\w#.,\' -]+\s+\b(?:street|st|road|rd|avenue|ave|drive|dr|'
        r'boulevard|blvd|lane|ln|court|ct|highway|hwy|way|parkway|pkwy)\b',
        re.I
    )
    city_state_zip_re = re.compile(r'\b[A-Z]{2}\s+\d{5}(?:-\d{4})?\b', re.I)
    address = None
    search_start = max(0, clinic_idx if clinic_idx is not None else 0)
    search_end = min(len(header), search_start + 9)
    for i in range(search_start, search_end):
        line = header[i]
        if excluded.search(line):
            continue
        if street_re.search(line):
            parts = [line]
            if i + 1 < len(header):
                nxt = header[i + 1]
                if city_state_zip_re.search(nxt) or re.search(r"\b[A-Z][a-z .'-]+,\s*[A-Z]{2}\s+\d{5}", nxt):
                    parts.append(nxt)
            address = ", ".join(parts)
            break

    # A second common layout prints street/city/state on one line.
    if not address:
        for line in header[search_start:search_end]:
            if excluded.search(line):
                continue
            if street_re.search(line) and (city_state_zip_re.search(line) or "," in line):
                address = line
                break

    return {"clinic_name": clinic_name, "clinic_address": address}



def _metadata_html_tables(text):
    """Return HTML table rows as plain cell strings for metadata-only inspection."""
    class _MetaTableParser(HTMLParser):
        def __init__(self):
            super().__init__(); self.tables=[]; self.in_table=False; self.rows=[]; self.row=None; self.cell=None
        def handle_starttag(self, tag, attrs):
            tag=tag.lower()
            if tag=="table":
                self.in_table=True; self.rows=[]
            elif self.in_table and tag=="tr":
                self.row=[]
            elif self.in_table and tag in ("td","th") and self.row is not None:
                self.cell=[]
        def handle_data(self, data):
            if self.in_table and self.cell is not None: self.cell.append(data)
        def handle_endtag(self, tag):
            tag=tag.lower()
            if tag in ("td","th") and self.cell is not None and self.row is not None:
                self.row.append(" ".join("".join(self.cell).split())); self.cell=None
            elif tag=="tr" and self.row is not None:
                if self.row: self.rows.append(self.row)
                self.row=None
            elif tag=="table" and self.in_table:
                if self.rows: self.tables.append(self.rows)
                self.in_table=False
    parser=_MetaTableParser()
    try: parser.feed(text or "")
    except Exception: pass
    return parser.tables
def extract_header_parties(text):
    """
    Extract client and clinic details from side-by-side HTML header cells.
    Does not modify pet, invoice number, or invoice date extraction.
    """
    result = {
        "clinic_name": None,
        "clinic_address": None,
        "client_name": None,
        "client_address": None
    }

    clinic_words = re.compile(
        r"\b(?:animal hospital|veterinary|vet clinic|pet hospital|"
        r"animal clinic|veterinary hospital)\b",
        re.I
    )

    street_re = re.compile(
        r"\b\d{1,6}\s+[\w#.,'-]+(?:\s+[\w#.,'-]+){0,8}\s+"
        r"(?:street|st|road|rd|avenue|ave|drive|dr|lane|ln|"
        r"boulevard|blvd|court|ct|way|parkway|pkwy)\b",
        re.I
    )

    city_zip_re = re.compile(
        r"\b[A-Z][A-Za-z .'-]+,\s*[A-Z]{2}\s+\d{5}(?:-\d{4})?\b"
    )

    def parse_block(value):
        value = re.sub(r"\s+", " ", str(value or "")).strip()
        street = street_re.search(value)

        if not street:
            return None, None

        city = city_zip_re.search(value, street.end())
        if not city:
            return None, None

        address = value[street.start():city.end()].strip(" ,")
        name = value[:street.start()].strip(" ,:-")

        # Remove common contact labels from the name portion.
        name = re.sub(
            r"\b(?:phone|fax|email|www\.com|www\.net)\b.*$",
            "",
            name,
            flags=re.I
        ).strip(" ,:-")

        return (name or None), address

    # Inspect cells in the same HTML table row because the two
    # address blocks are often printed side by side.
    for table in _metadata_html_tables(text):
        for row in table:
            parsed = [parse_block(cell) for cell in row]

            for index, cell in enumerate(row):
                if not clinic_words.search(cell):
                    continue

                clinic_name, clinic_address = parsed[index]

                if clinic_name:
                    result["clinic_name"] = clinic_name
                if clinic_address:
                    result["clinic_address"] = clinic_address

                # The adjacent non-clinic cell is the likely client block.
                for other_index, (other_name, other_address) in enumerate(parsed):
                    if other_index == index:
                        continue
                    if other_address and other_name:
                        result["client_name"] = other_name
                        result["client_address"] = other_address
                        break

                if result["clinic_name"] and result["client_name"]:
                    return result

    return result
def _metadata_table_values(text):
    """Find patient names and invoice-labelled values from HTML tables."""
    pets=[]
    invoice_dates=[]
    invoice_numbers=[]
    receipt_numbers=[]
    for table in _metadata_html_tables(text):
        for ridx,row in enumerate(table):
            headers=[re.sub(r"[^a-z]","",x.casefold()) for x in row]
            if any(x in {"patient","patientname","pet","petname","animal"} for x in headers):
                pidx=next(i for i,x in enumerate(headers) if x in {"patient","patientname","pet","petname","animal"})
                for data_row in table[ridx+1:]:
                    if pidx < len(data_row):
                        name=data_row[pidx].strip()
                        if _pet_ok(name) and name.casefold() not in {p.casefold() for p in pets}:
                            pets.append(name)
            row_text=" ".join(row)
            if re.search(r"\binvoice\s*(?:date|number|no\.?|#)\b",row_text,re.I):
                m = re.search(
                    r"\binvoice\s*date\b\s*[:#-]?\s*"
                    r"([A-Za-z]{3,9}\s+\d{1,2},?\s+\d{4}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}[/-]\d{1,2}[/-]\d{1,2})",
                    row_text,
                    re.IGNORECASE
                )
                if m:
                    invoice_dates.append(m.group(1))
                n=re.search(r"\binvoice\s*(?:number|no\.?|#)\b\s*[:#-]?\s*([A-Z0-9-]+)",row_text,re.I)
                if n: invoice_numbers.append(n.group(1))
            if re.search(r"\breceipt\s*(?:number|no\.?|#)\b",row_text,re.I):
                n=re.search(r"\breceipt\s*(?:number|no\.?|#)\b\s*[:#-]?\s*([A-Z0-9-]+)",row_text,re.I)
                if n: receipt_numbers.append(n.group(1))
    return pets, invoice_dates, invoice_numbers, receipt_numbers

def _labelled_invoice_date(text):
    """Prefer a date explicitly associated with Invoice Date/Date in an invoice block."""
    raw=re.sub(r"<[^>]+>","\n",text or "")
    lines=[re.sub(r"\s+"," ",x).strip() for x in raw.splitlines() if x.strip()]
    for i,line in enumerate(lines):
        if re.search(r"\binvoice\s+date\b",line,re.I):
            m=_DATE_TOKEN_RE.search(line)
            if m: return m.group(0)
            if i+1<len(lines):
                m=_DATE_TOKEN_RE.search(lines[i+1])
                if m: return m.group(0)
        # In a compact invoice-details table, Invoice Number and Date may share a row/block.
        if re.search(r"\binvoice\s*(?:number|no\.?|#)\b",line,re.I) and re.search(r"\bdate\b",line,re.I):
            m=list(_DATE_TOKEN_RE.finditer(line))
            if m: return m[-1].group(0)
        # Some invoices place "Invoice Number" and a standalone "Date" label
        # on adjacent rows of the same details block.
        if re.search(r"\binvoice\s*(?:number|no\.?|#)\b",line,re.I):
            for nearby in lines[i:i+4]:
                if re.fullmatch(r"date\s*:?", nearby, re.I):
                    j=lines.index(nearby, i)
                    if j+1 < len(lines):
                        m=_DATE_TOKEN_RE.search(lines[j+1])
                        if m: return m.group(0)
    return None
def extract_header_parties(text):
    """
    Extract client and clinic details from the two-column veterinary
    invoice/receipt header. Uses the first header block only.
    """
    lines = [
        re.sub(r"\s+", " ", line).strip()
        for line in str(text).splitlines()
        if line.strip()
    ]

    result = {
        "client_name": None,
        "client_address": None,
        "clinic_name": None,
        "clinic_address": None,
    }

    # Locate the beginning of the payment/invoice body.
    stop_patterns = [
        r"payment history",
        r"invoice number",
        r"receipt number",
        r"patient\s+provider",
    ]

    header_lines = []
    for line in lines:
        if any(re.search(p, line, re.I) for p in stop_patterns):
            break
        header_lines.append(line)

    # Split the two-column header using OCR spacing or table separators.
    left, right = [], []

    for line in header_lines:
        parts = re.split(r"\s{3,}|\s*\|\s*", line, maxsplit=1)

        if len(parts) == 2:
            if parts[0].strip():
                left.append(parts[0].strip())
            if parts[1].strip():
                right.append(parts[1].strip())
        else:
            # Preserve lines that OCR returned without column separation.
            left.append(line)

    def is_contact_or_address(line):
        return bool(re.search(
            r"\d{3}[-.)]\s*\d{3}[-.]\s*\d{4}|"
            r"\b(street|st|road|rd|lane|ln|avenue|ave|"
            r"drive|dr|boulevard|blvd|suite|fl|zip)\b|"
            r"@|www\.",
            line,
            re.I
        ))

    def clean_name(lines):
        candidates = [
            x for x in lines
            if not is_contact_or_address(x)
            and not re.search(r"\b(united states|usa)\b", x, re.I)
        ]
        return candidates[0] if candidates else None

    def address_lines(lines):
        return [
            x for x in lines
            if is_contact_or_address(x)
            or re.search(r"\b(united states|usa)\b", x, re.I)
        ]

    result["client_name"] = clean_name(left)
    result["clinic_name"] = clean_name(right)

    client_addr = address_lines(left)
    clinic_addr = address_lines(right)

    result["client_address"] = ", ".join(client_addr) if client_addr else None
    result["clinic_address"] = ", ".join(clinic_addr) if clinic_addr else None

    return result
def extract_general_metadata(text):
    flat=clean_ocr_text(text)
    header_meta = extract_clinic_header_metadata(text)
    party_meta = extract_header_parties(text)
    # Fallback: identify client and clinic from the document header
    header_fallback = extract_header_parties(text)

    for key in ["clinic_name", "clinic_address", "client_name", "client_address"]:
        if not header_meta.get(key) and header_fallback.get(key):
            header_meta[key] = header_fallback[key]
    parsed_dates = extract_all_dates(text)
    table_pets, table_invoice_dates, table_invoice_numbers, table_receipts = _metadata_table_values(text)
    extracted_pets, pet_evidence = extract_pet_names(text)
    all_pets=list(dict.fromkeys(extracted_pets + table_pets))
    # Invoice date must come from an invoice/date label where possible. Do not blindly
    # use the latest date: payment-entry dates and service dates may be later.
    labelled_date=_labelled_invoice_date(text)
    invoice_date=(table_invoice_dates[0] if table_invoice_dates else labelled_date)
    invoice_date_obj=_parse_document_date(invoice_date) if invoice_date else None
    if not invoice_date and parsed_dates:
        # Conservative fallback only when no explicit invoice/payment/service date labels exist.
        date_context=re.sub(r".{0,35}\b(?:payment|service|visit|treatment|receipt)\b.{0,35}"," ",flat,flags=re.I)
        candidates=extract_all_dates(date_context)
        if len(candidates)==1:
            invoice_date=candidates[0][1]; invoice_date_obj=candidates[0][0]

    invoice_number=first_match([
        r"\binvoice\s*(?:number|no\.?|#)\s*[:#-]?\s*([A-Z0-9-]+)"
    ],flat)
    if not invoice_number and table_invoice_numbers:
        invoice_number=table_invoice_numbers[0]
    receipt_number=first_match([
        r"\breceipt\s*(?:number|no\.?|#)\s*[:#-]?\s*([A-Z0-9-]+)"
    ],flat)
    if not receipt_number and table_receipts:
        receipt_number=table_receipts[0]

    clinic_address=header_meta["clinic_address"]
    if not clinic_address:
        # Letterhead blocks are often split into street and city/state/ZIP lines.
        raw_lines=[re.sub(r"\s+"," ",x).strip() for x in re.sub(r"<[^>]+>","\n",text or "").splitlines()]
        street_re=re.compile(r"\b\d{1,6}\s+[A-Za-z0-9 .#'-]+\b(?:St|Street|Rd|Road|Ave|Avenue|Dr|Drive|Ln|Lane|Blvd|Boulevard|Way|Ct|Court)\b",re.I)
        city_re=re.compile(r"\b[A-Z][A-Za-z .'-]+,\s*[A-Z]{2}\s+\d{5}(?:-\d{4})?\b")
        for i,line in enumerate(raw_lines):
            if street_re.search(line):
                parts=[line]
                for nxt in raw_lines[i+1:i+3]:
                    if city_re.search(nxt):
                        parts.append(nxt); break
                if len(parts)>1:
                    clinic_address=", ".join(parts); break

    meta={
        "clinic_name": header_meta["clinic_name"] or first_match([
            r"(?:clinic|hospital|provider)\s*[:#-]\s*([^|]{3,100})",
            r"^([^|]{3,100}?(?:animal hospital|veterinary|vet clinic|pet hospital))\b"
        ],flat),
        "clinic_address": clinic_address or first_match([
            r"(?:clinic\s+)?(?:address|location)\s*[:#-]\s*([^|]{5,180})",
            r"(\d{1,6}\s+[A-Za-z0-9 .'-]+,\s*[A-Za-z .'-]+,\s*[A-Z]{2}\s+\d{5}(?:-\d{4})?)"
        ],flat),
        "clinic_phone": first_match([r"(?:phone|ph|tel)\s*[:#-]?\s*([+()0-9 .-]{7,25})"],flat),
        "clinic_email": first_match([r"([A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,})"],flat),
        "client_name": first_match([r"(?:client|owner|pet owner|customer)\s*(?:name)?\s*[:#-]\s*([A-Z][A-Za-z .,'-]{2,70})"],flat),
        "client_address": first_match([r"(?:client|owner)\s+address\s*[:#-]\s*([^|]{5,140})"],flat),
        "pet_species": first_match([r"(?:species)\s*[:#-]\s*([A-Za-z -]{2,30})"],flat),
        "pet_breed": first_match([r"(?:breed)\s*[:#-]\s*([A-Za-z0-9 .'-]{2,50})"],flat),
        "pet_age": first_match([r"(?:age)\s*[:#-]\s*([A-Za-z0-9 .'-]{1,30})"],flat),
        "pet_sex": first_match([r"(?:sex|gender)\s*[:#-]\s*([A-Za-z -]{2,30})"],flat),
        "patient_id": first_match([r"(?:patient id|patient #|pet id)\s*[:#-]\s*([A-Z0-9-]{2,35})"],flat),
        "invoice_number": invoice_number,
        "receipt_number": receipt_number,
        "invoice_date": invoice_date,
        "invoice_date_iso": invoice_date_obj.isoformat() if invoice_date_obj else None,
        "all_document_dates": [token for _, token in parsed_dates],
        "service_dates": [token for _, token in parsed_dates if re.search(r"service|visit|treatment", flat, re.I)],
        "currency": "USD" if "$" in flat or re.search(r"\bUSD\b",flat,re.I) else None,
        "printed_total": first_match([r"(?:invoice total|grand total|total due|total)\s*[:$ ]+\$?([0-9,]+\.\d{2})"],flat),
        "pet_names": all_pets,
        "pet_names_evidence": (pet_evidence + [f"Patient table value: {p}" for p in table_pets])[:10]
    }
    return meta


# Optional Qwen3-4B metadata understanding.
# auto = run only if clinic/address/client is missing; off = disable; force = run for every page.
QWEN_MODE = "auto"  # "auto", "off", or "force"; keep off for reliable T4 PaddleOCR-VL runs
QWEN_MODEL_ID = "Qwen/Qwen3-4B"
QWEN_MAX_INPUT_CHARS = 4000
QWEN_DEBUG = True
_qwen_model = None
_qwen_tokenizer = None

def _load_qwen_metadata_model():
    """Lazy-load Qwen once. Prefer half precision on GPU to reduce memory use."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        if not torch.cuda.is_available():
            raise RuntimeError("Qwen GPU inference requested, but torch.cuda.is_available() is False.")
        _qwen_tokenizer = AutoTokenizer.from_pretrained(
            QWEN_MODEL_ID, trust_remote_code=True
        )
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_ID,
            torch_dtype=torch.float16,
            device_map={"": 0},
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )
        _qwen_model.eval()
    return _qwen_model, _qwen_tokenizer

def _normalise_evidence(value):
    return re.sub(r"\s+", " ", str(value or "")).strip().casefold()

def _extract_json_object(answer):
    """Parse a JSON object even if Qwen surrounds it with explanation/think text."""
    import json
    answer = re.sub(r"<think>.*?</think>", "", answer or "", flags=re.S).strip()
    decoder = json.JSONDecoder()
    for pos, char in enumerate(answer):
        if char == "{":
            try:
                obj, _ = decoder.raw_decode(answer[pos:])
                if isinstance(obj, dict):
                    return obj
            except Exception:
                continue
    raise ValueError("No valid JSON object found in Qwen response: " + answer[:350])

def _qwen_json(system, user, max_new_tokens=450):
    """One deterministic Qwen call (the ORCA rules are the system prompt). Returns the parsed JSON object."""
    import torch
    model, tokenizer = _load_qwen_metadata_model()
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt")
    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                                   pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return _extract_json_object(answer)


QWEN_BILLING_CLASSES = {"INVOICE", "PHARMACY", "ONLINE_PROVIDER"}

def qwen_metadata_fallback(metadata, source_text, document_class=None):
    """
    Interpret relationships between clinic header, clinic address, Bill To/client block and pet labels.
    Python values are preserved. Qwen values are accepted only with OCR-backed (verbatim) evidence.
    """
    required = ["clinic_name", "clinic_address", "client_name", "invoice_number", "invoice_date"]
    missing = [k for k in required if not metadata.get(k)]
    pets_missing = not metadata.get("pet_names")
    if QWEN_MODE not in {"auto", "off", "force"}:
        raise ValueError("QWEN_MODE must be 'auto', 'off', or 'force'")
    if QWEN_MODE == "off" or (QWEN_MODE == "auto" and not missing and not pets_missing):
        metadata["qwen_metadata_status"] = "SKIPPED_NOT_NEEDED" if not (missing or pets_missing) else "DISABLED"
        metadata["qwen_metadata_missing_before_qwen"] = missing
        return metadata

    if not (source_text or "").strip():
        metadata["qwen_metadata_status"] = "NO_OCR_TEXT"
        metadata["qwen_metadata_missing_before_qwen"] = missing
        return metadata

    # Qwen is optional. Never let unavailable CUDA PyTorch interrupt primary OCR/export.
    try:
        import torch as _qwen_torch
        if not _qwen_torch.cuda.is_available():
            metadata["qwen_metadata_status"] = "SKIPPED_QWEN_GPU_UNAVAILABLE"
            metadata["qwen_metadata_missing_before_qwen"] = missing
            metadata["qwen_note"] = "Qwen requires a separate CUDA-enabled runtime or inference endpoint; PaddleOCR-VL GPU processing continues."
            return metadata
    except Exception as _qwen_import_error:
        metadata["qwen_metadata_status"] = "SKIPPED_QWEN_RUNTIME_UNAVAILABLE"
        metadata["qwen_metadata_missing_before_qwen"] = missing
        metadata["qwen_note"] = str(_qwen_import_error)[:300]
        return metadata

    prompt = f"""
Task: metadata extraction for one page (document class: {document_class or 'unknown'}).

Return ONLY valid JSON with exactly this structure:
{{
  "clinic_name": string or null,
  "clinic_address": string or null,
  "client_name": string or null,
  "invoice_number": string or null,
  "invoice_date": string or null,
  "pet_names": [string] ,
  "evidence": {{
    "clinic_name": "exact OCR phrase supporting clinic_name" or null,
    "clinic_address": "exact OCR phrase supporting clinic_address" or null,
    "client_name": "exact OCR phrase supporting client_name" or null,
    "invoice_number": "exact OCR phrase supporting invoice_number" or null,
    "invoice_date": "exact OCR phrase supporting invoice_date" or null,
    "pet_names": "one exact OCR phrase that contains the pet name(s), e.g. the Animal / Patient / Prescribed For label and value" or null
  }}
}}

Notes:
- clinic_name is the clinic/pharmacy/online provider that issued the document (often the letterhead); clinic_address is ITS address, never the Bill To/customer address.
- client_name is the person in the Bill To / Client / Owner / Customer section. Never put the client's name into pet_names.
- pet_names: only names printed next to a pet label (Pet Name, Patient Name, Patient, Animal, Prescribed For). Do not infer a pet from a medication. Use an empty list if none is printed.
- Use only information explicitly visible in the OCR text; otherwise null. Evidence must be copied verbatim from the OCR text.

OCR TEXT:
{source_text[:QWEN_MAX_INPUT_CHARS]}
""".strip()

    try:
        result = _qwen_json(QWEN_SYSTEM_PROMPT, prompt, 220)
        evidence = result.get("evidence") if isinstance(result.get("evidence"), dict) else {}
        source_norm = _normalise_evidence(source_text)
        accepted = []

        for field in required:
            value = result.get(field)
            quote = evidence.get(field)
            if metadata.get(field):
                continue  # deterministic extraction always wins
            if not isinstance(value, str) or not value.strip():
                continue
            if not isinstance(quote, str) or not quote.strip():
                continue
            quote_norm = _normalise_evidence(quote)
            if quote_norm and quote_norm in source_norm:
                metadata[field] = value.strip()
                metadata[field + "_qwen_evidence"] = quote.strip()
                accepted.append(field)

        if pets_missing or QWEN_MODE == "force":
            new_pets = validate_qwen_pet_names(result.get("pet_names"), evidence.get("pet_names"), source_text, metadata.get("client_name"))
            new_pets = [n for n in new_pets if n.casefold() not in {x.casefold() for x in metadata.get("pet_names", [])}]
            if new_pets:
                metadata["pet_names"] = list(metadata.get("pet_names", [])) + new_pets
                metadata["pet_names_qwen_evidence"] = str(evidence.get("pet_names")).strip()
                accepted.append("pet_names")

        metadata["qwen_metadata_status"] = ("COMPLETED_FIELDS_ACCEPTED" if accepted else "COMPLETED_NO_VERIFIED_FIELDS")
        metadata["qwen_metadata_fields_accepted"] = accepted
        metadata["qwen_metadata_missing_after_qwen"] = [k for k in required if not metadata.get(k)]
        if QWEN_DEBUG:
            print("[Qwen metadata]", metadata.get("source_file", ""), "status=", metadata["qwen_metadata_status"],
                  "accepted=", accepted, "still_missing=", metadata["qwen_metadata_missing_after_qwen"])
    except Exception as exc:
        metadata["qwen_metadata_status"] = "FAILED_REVIEW_REQUIRED"
        metadata["qwen_metadata_error"] = f"{type(exc).__name__}: {str(exc)[:500]}"
        metadata["qwen_metadata_missing_after_qwen"] = [k for k in required if not metadata.get(k)]
        if QWEN_DEBUG:
            print("[Qwen metadata ERROR]", metadata["qwen_metadata_error"])
    return metadata


# Qwen line-item fallback: used ONLY when no deterministic parser (HTML table, header-aware text, order-card) found a row on a billing page.
# Every accepted row is checked by validate_qwen_line_items (verbatim evidence, printed amount, words present) and is always review_required.
QWEN_LINE_ITEM_FALLBACK = False  # optional; enable only in a separate compatible Qwen runtime
qwen_line_item_log = []

def qwen_line_item_fallback(source_text, filename, page, known_pets=(), document_class=None):
    entry = {"source_file": filename, "page_number": page, "status": None, "accepted": 0, "rejected": []}
    qwen_line_item_log.append(entry)
    if QWEN_MODE == "off" or not QWEN_LINE_ITEM_FALLBACK:
        entry["status"] = "DISABLED"; return []
    if not (source_text or "").strip():
        entry["status"] = "NO_OCR_TEXT"; return []
    prompt = f"""
Task: list the purchased line items on this {document_class or 'billing'} page (veterinary service, medication or product rows only).

Return ONLY valid JSON:
{{
  "line_items": [
    {{"description": string, "quantity": string or null, "allowed": "extended line amount exactly as printed on that row, e.g. 163.08",
      "pet_name": string or "UNKNOWN", "evidence": "the row's own OCR text, copied verbatim, containing the description, quantity and amount"}}
  ],
  "uncertainty_flags": [string]
}}

Notes:
- allowed is the extended line total of the row, never a unit price, subtotal, total, tax, shipping, discount or payment.
- Do not list totals, tax, shipping, discounts, payments, thank-you text, staff names or prescription instructions as items.
- Do not guess: if a row is unclear, leave it out and describe it in uncertainty_flags.

OCR TEXT:
{source_text[:QWEN_MAX_INPUT_CHARS]}
""".strip()
    try:
        result = _qwen_json(QWEN_SYSTEM_PROMPT, prompt, 350)
        accepted, rejected = validate_qwen_line_items(result, source_text, known_pets)
        entry.update({"status": "COMPLETED", "accepted": len(accepted), "rejected": rejected,
                      "uncertainty_flags": result.get("uncertainty_flags") if isinstance(result, dict) else None})
        out = []
        for a in accepted:
            out.append({"description": a["description"], "quantity": a["quantity"], "allowed_amount": a["allowed_amount"], "pet_name": a["pet_name"],
                        "service_date": None, "row_tax": None, "line_type": "item", "source_file": filename, "page_number": page,
                        "source_text": a["evidence_quote"], "evidence_line_number": None, "extraction_method": "qwen_line_item_fallback",
                        "review_required": True, "review_reasons": a["review_reasons"]})
        if QWEN_DEBUG:
            print("[Qwen line items]", filename, "p", page, "accepted=", len(accepted), "rejected=", len(rejected))
        return out
    except Exception as exc:
        entry["status"] = "FAILED_REVIEW_REQUIRED"; entry["error"] = f"{type(exc).__name__}: {str(exc)[:500]}"
        if QWEN_DEBUG:
            print("[Qwen line items ERROR]", entry["error"])
        return []


# Collect text per source file from native PDF text and PaddleOCR-VL output.
for filename in sorted({p["source_file"] for p in all_pages}):
    file_pages=[p for p in all_pages if p["source_file"]==filename]
    file_runs=[r for r in page_runs if r["source_file"]==filename]
    page_records=[]
    for p in file_pages:
        run=next((r for r in file_runs if r["page_number"]==p["page_number"]),{})
        ocr_text="\n".join([p.get("native_text","")] + collect_text(run.get("raw_output",[])))
        cls_result=classification_by_page.get((filename,p["page_number"]),{})
        meta=extract_general_metadata(ocr_text)
        _pn,_pev=extract_pet_names(ocr_text)
        meta["pet_names"]=list(dict.fromkeys(list(meta.get("pet_names") or [])+_pn)) if not _pn else _pn
        if _pev: meta["pet_names_evidence"]=_pev[:5]
        if meta.get("clinic_name") and re.search(r"^\d{1,2}:\d{2}|\b(?:Placed|Delivered)\b",meta["clinic_name"]):
            meta["clinic_name"]=None   # phone status bar / order-status text is not a provider name
        if meta.get("invoice_number") and not re.search(r"\d",meta["invoice_number"]):
            meta["invoice_number"]=None   # a label word (e.g. a column heading) is not an invoice number
        if not meta.get("invoice_number"):
            _o=re.search(r"\border\s*(?:number|no\.?|#|id)\s*[:#-]?\s*((?=[A-Z0-9-]*\d)[A-Z0-9][A-Z0-9-]{4,})",clean_ocr_text(ocr_text),re.I)
            if _o:
                meta["invoice_number"]=_o.group(1); meta["invoice_number_source"]="order_number"
        meta.pop("veterinarian_name", None)
        meta=qwen_metadata_fallback(meta, ocr_text, cls_result.get("final_class"))
        meta.update({
            "source_file":filename,"page_number":p["page_number"],"page_count":p.get("page_count",len(file_pages)),
            "document_class":cls_result.get("final_class","CLASSIFICATION_FAILED"),
            "document_display_name":cls_result.get("display_label", "Online Store" if cls_result.get("final_class")=="ONLINE_PROVIDER" else cls_result.get("final_class","Classification failed")),
            "processing_type":cls_result.get("processing_type"),
            "model_class":cls_result.get("model_class"),"classification_confidence":cls_result.get("confidence"),
            "classification_decision":cls_result.get("decision"),"classification_review_required":cls_result.get("review_required",True),
            "classification_rule":cls_result.get("rule_applied"),"metadata_source_text":ocr_text[:4000]
        })
        page_records.append(meta)

    # Aggregate document-level fields across pages of the SAME uploaded file.
    # This handles letterhead on page 1 and billing details/line items on later pages.
    _doc_text_for_dates = "\\n".join(
        [p.get("native_text", "") for p in file_pages] +
        ["\\n".join(collect_text(r.get("raw_output", []))) for r in file_runs]
    )
    _doc_dates = extract_all_dates(_doc_text_for_dates)
    _latest_doc_date = _doc_dates[-1] if _doc_dates else (None, None)
    _shared_fields = [
        "clinic_name", "clinic_address", "clinic_phone", "clinic_email",
        "client_name", "client_address"
    ]
    for _field in _shared_fields:
        _value = next((m.get(_field) for m in page_records if m.get(_field)), None)
        if _value:
            for _m in page_records:
                if not _m.get(_field):
                    _m[_field] = _value
                    _m[_field + "_aggregation_source"] = "same_document_other_page"
    _all_pets = list(dict.fromkeys(
        pet for m in page_records for pet in (m.get("pet_names") or [])
    ))
    if _all_pets:
        for _m in page_records:
            _m["pet_names"] = _all_pets
    if _latest_doc_date[0]:
        for _m in page_records:
            _m["invoice_date"] = _latest_doc_date[1]
            _m["invoice_date_iso"] = _latest_doc_date[0].isoformat()
            _m["all_document_dates"] = [token for _, token in _doc_dates]

    all_document_metadata.extend(page_records)

    # v4.6: all pages of all document classes are considered for item extraction.
    eligible_pages=[]
    for p in file_pages:
        cr=classification_by_page.get((filename,p["page_number"]),{})
        final_class=cr.get("final_class")
        eligible_pages.append(p)   # v4.6: every document class is parsed for line items (no INVOICE/PHARMACY/ONLINE_PROVIDER gate); the class stays in the output
    # Pages with tables are included later after PaddleOCR table detection, even if their class differs.
    # Metadata extraction above is unconditional for all document classes.
    file_start=len(rows)
    file_pets=list(dict.fromkeys(n for m in page_records for n in (m.get("pet_names") or [])))
    for p in file_pages:
        run=next((r for r in file_runs if r["page_number"]==p["page_number"]),{})
        page_tables=parse_html_invoice_tables(run.get("raw_output",[]),filename,p["page_number"],"paddleocr_vl15_html_table")
        page_meta=next((m for m in page_records if m["page_number"]==p["page_number"]),{})
        log={"source_file":filename,"page_number":p["page_number"],"eligible":p in eligible_pages,"html_table_rows":len(page_tables),"text_rows":0,"qwen_rows":0,"source":None}
        if page_tables:
            rows.extend(page_tables); log["source"]="paddleocr_vl15_html_table"
        elif p in eligible_pages:
            # No HTML table: pharmacy / online-order / text-only invoices. Deterministic text parsers first, Qwen only if they find nothing.
            _items,_taxes,_info=extract_rows_from_page_text(p,run,filename,"text")
            if _items or _taxes:
                rows.extend(_items); rows.extend(_taxes)
                log.update({"text_rows":len(_items),"source":_info["source"],"unattached_lines":_info.get("unattached_lines")})
            else:
                _q=qwen_line_item_fallback(page_meta.get("metadata_source_text") or "\n".join([p.get("native_text","")]+collect_text(run.get("raw_output",[]))),
                                           filename,p["page_number"],file_pets,page_meta.get("document_class"))
                rows.extend(_q); log.update({"qwen_rows":len(_q),"source":"qwen_line_item_fallback" if _q else None})
            if classification_by_page.get((filename,p["page_number"]),{}).get("rule_applied")=="R3_CONFLICT_PHARMACY_ONLINE":
                for r in rows[file_start:]:
                    if r["page_number"]==p["page_number"]:
                        r["review_required"]=True; r["review_reasons"]=list(dict.fromkeys(list(r.get("review_reasons",[]))+["pharmacy_online_class_conflict"]))
        page_extraction_log.append(log)

    # A document that names exactly one pet: rows without their own pet label belong to that pet (recorded, never guessed across several pets).
    if len(file_pets)==1:
        for r in rows[file_start:]:
            if not r.get("pet_name"):
                r["pet_name"]=file_pets[0]; r["pet_assignment_source"]="single_pet_named_in_document"

    # Preserve invoice-level totals as evidence only; never convert totals/payments to line items.
    combined="\n".join([p.get("native_text","") for p in file_pages] + ["\n".join(collect_text(r.get("raw_output",[]))) for r in file_runs])
    flat=clean_ocr_text(combined)
    # Capture explicit invoice totals and patient subtotals as reconciliation evidence.
    summary_pattern = re.compile(
        r"\b(Patient\s+Subtotal|Invoice\s+Total|Grand\s+Total|Total\s+Due|Order\s+Total|Items?\s+Subtotal|Sub-?total|Total)\b"
        r"\s*[:$ ]*\$?([0-9,]+\.\d{2})",
        re.I
    )
    for total_match in summary_pattern.finditer(flat):
        invoice_summaries.append({
            "source_file": filename,
            "label": re.sub(r"\s+", " ", total_match.group(1)).strip(),
            "amount": money(dec(total_match.group(2))),
            "source_text": total_match.group(0)
        })

    # Shipping and discounts: audit metadata only (deduplicated because native text and OCR text repeat the same lines).
    for adj in extract_adjustments(combined, filename):
        k=(adj["source_file"],adj["kind"],adj["label"].lower(),adj["amount"])
        if k not in {(a["source_file"],a["kind"],a["label"].lower(),a["amount"]) for a in invoice_adjustments}:
            invoice_adjustments.append(adj)

# Tax rows from explicit numeric tax columns only; never infer or manufacture tax.
for r in rows:
    row_tax = r.pop("row_tax", None)
    # Blank cells, a taxable marker such as "T", and 0.00 do not create a Tax item.
    if row_tax is not None and row_tax != 0:
        tax_rows.append({
            "description": "Tax", "quantity": 1, "allowed_amount": money(row_tax),
            "pet_name": r.get("pet_name"), "service_date": None, "line_type": "tax",
            "source_file": r["source_file"], "page_number": r["page_number"],
            "source_text": "Tax column: " + r["source_text"],
            "evidence_line_number": r.get("evidence_line_number"),
            "extraction_method": r["extraction_method"],
            "review_required": False, "review_reasons": []
        })
rows.extend(tax_rows)

# Per-pet grouping: every pet gets a separate claim-preparation group; unknown assignments stay UNKNOWN.
for r in rows:
    r["pet_name"] = r.get("pet_name") or "UNKNOWN"
    if r["pet_name"] == "UNKNOWN":
        r["review_required"] = True
        r["review_reasons"] = list(dict.fromkeys(r.get("review_reasons",[])+["pet_assignment_not_explicitly_identified"]))
    key=f'{r["source_file"]}|{r["page_number"]}|{r.get("source_text","")}|{r["description"]}|{r["pet_name"]}'
    r["line_item_id"]=hashlib.sha1(key.encode()).hexdigest()[:12]

EVIDENCE_COLUMNS=["line_item_id","pet_name","pet_assignment_source","description","quantity","allowed_amount","line_type","service_date","source_file","page_number","source_text","evidence_line_number","extraction_method","review_required","review_reasons"]
line_items_df=pd.DataFrame(rows,columns=EVIDENCE_COLUMNS)

# The compact Orca payload still has exactly the three requested fields.
orca_payload=line_items_df[["description","quantity","allowed_amount"]].rename(columns={"allowed_amount":"allowed"})

# Nested per-document/per-pet view: each pet's metadata is followed by that pet's own line-item table.
pet_claims=[]
for filename in sorted({r["source_file"] for r in rows}):
    file_rows=line_items_df[line_items_df.source_file==filename]
    meta_rows=[m for m in all_document_metadata if m["source_file"]==filename]
    pets=sorted(file_rows.pet_name.dropna().unique().tolist())
    for pet in pets:
        pet_rows=file_rows[file_rows.pet_name==pet]
        pet_claims.append({"source_file":filename,"pet_name":pet,"document_classes":sorted({m["document_class"] for m in meta_rows}),"line_items":pet_rows[["description","quantity","allowed_amount"]].rename(columns={"allowed_amount":"allowed"}).to_dict("records"),"review_required":bool(pet_rows.review_required.fillna(True).any())})

invoice_metadata_df=pd.DataFrame(all_document_metadata).reset_index(drop=True)
invoice_metadata_df.index = range(1, len(invoice_metadata_df) + 1)
classification_df=pd.DataFrame([{k:v for k,v in r.items() if k not in {"probabilities","evidence","reasons"}} for r in classification_results])
display(classification_df[[c for c in ["source_file","page_number","predicted_class","final_class","confidence","processing_type","decision"] if c in classification_df.columns]])
display(invoice_metadata_df[[c for c in ["source_file","page_number","document_class","clinic_name","clinic_address","client_name","pet_names","invoice_number","invoice_date"] if c in invoice_metadata_df.columns]])
for claim in pet_claims:
    print("\nPET:",claim["pet_name"],"| FILE:",claim["source_file"])
    _pet_df = pd.DataFrame(claim["line_items"], columns=["description", "quantity", "allowed"])
    _pet_df.index = range(1, len(_pet_df) + 1)
    display(_pet_df)

,source_file,page_number,predicted_class,final_class,confidence,processing_type,decision
0,23026778.pdf,1,INVOICE,INVOICE,0.98,INVOICE,RULE_CLASSIFIED


,source_file,page_number,document_class,clinic_name,clinic_address,client_name,pet_names,invoice_number,invoice_date
1,23026778.pdf,1,INVOICE,None,None,None,"[Abbey, Auggie]",54970,10/15/2027



PET: Abbey | FILE: 23026778.pdf


,description,quantity,allowed
1,Examination,1,65.00
2,Bordetella Oral,1,28.00
3,Bordetella Oral,1,0.00
4,Clindamycin 300Mg,28,59.80
5,Urinalysis,1,28.00
6,Prednisone 20Mg,14,21.30



PET: Auggie | FILE: 23026778.pdf


,description,quantity,allowed
1,Bordetella Oral,1,0.00
2,Bordetella Oral,1,28.00


## 9. Reconcile allowed amounts against invoice total

Sum genuine medical line-item allowed amounts plus separately extracted non-zero tax and compare with the explicit Invoice Total. Payment History / Amount Paid / Payment Visa / Paid to Date are never treated as invoice line items or invoice totals. Do not force a match; discrepancies require review.


In [16]:
def grand_total_for(sums):
    """Prefer a payable total; fall back to Patient Subtotal."""
    priority = {
        "total due": 0,
        "grand total": 1,
        "invoice total": 2,
        "order total": 3,
        "total": 4,
        "patient subtotal": 5,
        "items subtotal": 6,
        "item subtotal": 6,
        "subtotal": 7,
        "sub-total": 7,
    }
    # (Patient / Items) Subtotal are item-only amounts; the others are payable totals.
    candidates = [
        item for item in sums
        if str(item.get("label", "")).strip().lower() in priority
    ]
    return min(
        candidates,
        key=lambda item: priority[str(item["label"]).strip().lower()]
    ) if candidates else None


recon = []
all_files = sorted(
    set(line_items_df.source_file.dropna()) |
    {item["source_file"] for item in invoice_summaries}
)

for filename in all_files:
    df = line_items_df[line_items_df.source_file == filename]

    item_sum = sum(
        (dec(value) or Decimal("0.00")
         for value in df.loc[df.line_type == "item", "allowed_amount"].dropna()),
        Decimal("0.00")
    )
    tax_sum = sum(
        (dec(value) or Decimal("0.00")
         for value in df.loc[df.line_type == "tax", "allowed_amount"].dropna()),
        Decimal("0.00")
    )

    totalrec = grand_total_for([
        item for item in invoice_summaries if item["source_file"] == filename
    ])
    printed = dec(totalrec["amount"]) if totalrec else None
    label = str(totalrec["label"]).strip().lower() if totalrec else None

    # Patient Subtotal is compared with item rows only.
    # Grand/Invoice Total/Total Due includes separately extracted tax.
    # Shipping printed on the document is part of an online/pharmacy order total (audit metadata, not a line item).
    shipping_sum = sum((dec(a["amount"]) or Decimal("0.00") for a in invoice_adjustments
                        if a["source_file"] == filename and a["kind"] == "shipping"), Decimal("0.00"))
    if label in {"patient subtotal", "items subtotal", "item subtotal", "subtotal", "sub-total"}:
        comparison_amount = item_sum
        comparison_basis = "items_only_vs_subtotal"
    else:
        comparison_amount = item_sum + tax_sum + shipping_sum
        comparison_basis = "items_plus_tax_plus_shipping_vs_payable_total"

    difference = comparison_amount - printed if printed is not None else None

    has_discount = any(
        re.search(r"\b(discount|coupon|credit|refund)\b", str(value), re.I)
        for value in df.source_text.fillna("")
    )

    reasons = []
    if printed is None:
        reasons.append("printed_total_or_patient_subtotal_not_found")
    elif difference != Decimal("0.00"):
        reasons.append("invoice_sum_mismatch")
    if has_discount:
        reasons.append("discount_or_credit_requires_review")

    status = (
        "MATCH"
        if printed is not None and difference == Decimal("0.00") and not has_discount
        else "REVIEW"
    )

    recon.append({
        "source_file": filename,
        "items_sum": money(item_sum),
        "tax_sum": money(tax_sum),
        "shipping_sum": money(shipping_sum),
        "calculated_total": money(item_sum + tax_sum),
        "comparison_calculated_amount": money(comparison_amount),
        "printed_comparison_amount": money(printed),
        "printed_amount_label": totalrec["label"] if totalrec else None,
        "comparison_basis": comparison_basis,
        "difference": money(difference),
        "status": status,
        "review_reasons": reasons
    })

    # Do not auto-correct extracted values to force a match.
    # Flag all rows from a mismatching invoice for human review.
    if status == "REVIEW" and len(df):
        mask = line_items_df.source_file == filename
        line_items_df.loc[mask, "review_required"] = True
        for row_idx in line_items_df.index[mask]:
            existing = line_items_df.at[row_idx, "review_reasons"]
            if not isinstance(existing, list):
                existing = [] if existing is None else [str(existing)]
            line_items_df.at[row_idx, "review_reasons"] = list(dict.fromkeys(
                existing + reasons
            ))

# Update pet-level review status after invoice reconciliation.
for claim in pet_claims:
    file_rows = line_items_df[
        (line_items_df.source_file == claim["source_file"]) &
        (line_items_df.pet_name == claim["pet_name"])
    ]
    claim["review_required"] = bool(file_rows.review_required.fillna(True).any())

reconciliation_df = pd.DataFrame(recon)
display(reconciliation_df)


,source_file,items_sum,tax_sum,shipping_sum,calculated_total,comparison_calculated_amount,printed_comparison_amount,printed_amount_label,comparison_basis,difference,status,review_reasons
0,23026778.pdf,230.10,0.00,0.00,230.10,230.10,230.10,Invoice Total,items_plus_tax_plus_shipping_vs_payable_total,0.00,MATCH,[]


## 10. Export review and audit files

The compact CSV is only the three requested fields. The audit JSON retains source evidence, extraction method, and review reasons. Reconciliation failures promote all rows from that file to review.

## 11. BPI routing and output-contract checks

These tests do not run model inference. They verify the deterministic class-to-processing mapping and final Orca payload contract.


In [17]:
# Synthetic rule tests only; these do not replace validation on real claim documents.
_test_cases = [
    ("Claim Form Claimant Policy Number Invoice Subtotal Total Qty Amount $65.00 $28.00", "INVOICE"),
    ("Pharmacy Prescription Rx Number Dispensed Medication Refills", "PHARMACY"),
    ("Order Number Shipping Tracking Order Summary Items Ordered https://shop.example", "ONLINE_PROVIDER"),
    ("Treatment Plan Estimate Proposed treatment LOW $100.00 HIGH $200.00", "ESTIMATE"),
    ("Medical Record History Diagnosis Assessment Physical Exam", "MEDICAL_RECORD"),
    ("Claim Form Claimant Policy Number Insured Pet Name", "CLAIM_FORM"),
    ("Payment Slip Visa Amount Tendered $50.00", "INCOMPLETE"),
]
for _text, _expected in _test_cases:
    _got=classify_bpi_text(_text,submitted_pages=1)['final_class']
    assert _got==_expected, (_expected,_got,_text)
print('Synthetic BPI rule tests passed. These tests validate rule wiring, not model accuracy on real documents.')

Synthetic BPI rule tests passed. These tests validate rule wiring, not model accuracy on real documents.


In [18]:
# Deterministic contract checks (safe to run after extraction)
EXPECTED_CLASSES = {
    "CLAIM_FORM", "INVOICE", "MEDICAL_RECORD", "ESTIMATE",
    "PHARMACY", "ONLINE_PROVIDER", "INCOMPLETE", "OTHER_UNKNOWN"
}
def bpi_processing_type(label):
    return "INVOICE" if label in {"PHARMACY", "ONLINE_PROVIDER"} else label

assert bpi_processing_type("PHARMACY") == "INVOICE"
assert bpi_processing_type("ONLINE_PROVIDER") == "INVOICE"
assert bpi_processing_type("MEDICAL_RECORD") == "MEDICAL_RECORD"
assert list(orca_payload.columns) == ["description", "quantity", "allowed"]
assert set(classification_df["final_class"].dropna()).issubset(
    EXPECTED_CLASSES | {"CLASSIFICATION_FAILED"}
)
print("BPI class mapping and Orca three-field payload checks passed.")
print("Canonical ONLINE_PROVIDER label is preserved; display name is Online Store.")


BPI class mapping and Orca three-field payload checks passed.
Canonical ONLINE_PROVIDER label is preserved; display name is Online Store.


In [20]:

# ============================================================
# FINAL EXPORT: TWO CONSOLIDATED OUTPUT FILES
# 1. all_extracted_metadata.json
# 2. pet_wise_line_items.json
# ============================================================

import json
import zipfile
from pathlib import Path
import pandas as pd

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. CONSOLIDATED DOCUMENT METADATA
# ------------------------------------------------------------

metadata_records = []

for record in all_document_metadata:
    metadata_records.append(dict(record))

# Add classification details to the corresponding metadata record.
classification_lookup = {}

if len(classification_df):
    for _, row in classification_df.iterrows():
        key = (
            str(row.get("source_file", "")),
            str(row.get("page_number", ""))
        )
        classification_lookup[key] = row.to_dict()

for record in metadata_records:
    key = (
        str(record.get("source_file", "")),
        str(record.get("page_number", ""))
    )

    classification = classification_lookup.get(key, {})

    if classification:
        record["document_classification"] = classification.get(
            "final_class",
            classification.get("predicted_class")
        )
        record["classification_confidence"] = classification.get(
            "confidence"
        )
        record["processing_type"] = classification.get("processing_type")
        record["decision"] = classification.get("decision")

# Add invoice summaries and adjustments as supporting evidence.
metadata_output = {
    "total_documents": len(metadata_records),
    "documents": metadata_records,
    "invoice_summaries": invoice_summaries,
    "invoice_adjustments": invoice_adjustments
}

metadata_path = OUTPUT_DIR / "all_extracted_metadata.json"

metadata_path.write_text(
    json.dumps(
        metadata_output,
        ensure_ascii=False,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 2. PET-WISE LINE ITEMS
# ------------------------------------------------------------

pet_wise_output = []

for claim in pet_claims:

    pet_name = claim.get("pet_name", "UNKNOWN")
    source_file = claim.get("source_file")

    items = []

    for item in claim.get("line_items", []):

        items.append({
            "description": item.get("description"),
            "quantity": item.get("quantity"),
            "allowed": item.get("allowed")
        })

    pet_wise_output.append({
        "pet_name": pet_name,
        "source_file": source_file,
        "line_item_count": len(items),
        "line_items": items
    })

pet_items_path = OUTPUT_DIR / "pet_wise_line_items.json"

pet_items_path.write_text(
    json.dumps(
        pet_wise_output,
        ensure_ascii=False,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 3. OPTIONAL REVIEW FILE
# ------------------------------------------------------------

review = (
    line_items_df[line_items_df["review_required"] == True].copy()
    if len(line_items_df) and "review_required" in line_items_df.columns
    else pd.DataFrame(columns=EVIDENCE_COLUMNS)
)

review_path = OUTPUT_DIR / "human_review_required.json"

review.to_json(
    review_path,
    orient="records",
    indent=2,
    force_ascii=False
)

# ------------------------------------------------------------
# 4. CREATE ZIP WITH THE TWO MAIN FILES + REVIEW FILE
# ------------------------------------------------------------

zip_path = ROOT / "orca_extracted_claim_data.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:

    z.write(
        metadata_path,
        arcname="all_extracted_metadata.json"
    )

    z.write(
        pet_items_path,
        arcname="pet_wise_line_items.json"
    )

    z.write(
        review_path,
        arcname="human_review_required.json"
    )

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

print("EXPORT COMPLETED")
print("----------------------------------------")
print("Metadata records:", len(metadata_records))
print("Pet groups:", len(pet_wise_output))
print("Total pet-wise line items:", sum(
    x["line_item_count"] for x in pet_wise_output
))
print("Items requiring review:", len(review))
print("----------------------------------------")
print("Metadata file:", metadata_path)
print("Pet-wise line items file:", pet_items_path)
print("ZIP file:", zip_path)

# Download consolidated ZIP
from google.colab import files
files.download(str(zip_path))


EXPORT COMPLETED
----------------------------------------
Metadata records: 1
Pet groups: 2
Total pet-wise line items: 8
Items requiring review: 0
----------------------------------------
Metadata file: /content/orca_vet_line_items/outputs/all_extracted_metadata.json
Pet-wise line items file: /content/orca_vet_line_items/outputs/pet_wise_line_items.json
ZIP file: /content/orca_vet_line_items/orca_extracted_claim_data.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
review=line_items_df[line_items_df.review_required==True].copy() if len(line_items_df) else pd.DataFrame(columns=EVIDENCE_COLUMNS)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
orca_payload.to_csv(OUTPUT_DIR/'orca_minimal_line_items.csv',index=False)
line_items_df.to_json(OUTPUT_DIR/'line_items_with_evidence.json',orient='records',indent=2,force_ascii=False)
pd.DataFrame(all_document_metadata).to_json(OUTPUT_DIR/'all_document_metadata.json',orient='records',indent=2,force_ascii=False)
classification_df.to_json(OUTPUT_DIR/'document_classification.json',orient='records',indent=2,force_ascii=False)
# Human-readable classification summary; canonical class labels remain unchanged.
classification_display = classification_df.copy()
if "display_label" not in classification_display.columns:
    classification_display["display_label"] = classification_display.get("final_class")
classification_display.to_csv(OUTPUT_DIR/'document_classification_summary.csv', index=False)
Path(OUTPUT_DIR/'pet_grouped_claims.json').write_text(json.dumps(pet_claims,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
pd.DataFrame(invoice_summaries).to_csv(OUTPUT_DIR/'invoice_summary_evidence.csv',index=False)
pd.DataFrame(invoice_adjustments).to_csv(OUTPUT_DIR/'invoice_adjustments_audit.csv',index=False)   # shipping / discounts (not line items)
Path(OUTPUT_DIR/'qwen_line_item_fallback_log.json').write_text(json.dumps(qwen_line_item_log,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
Path(OUTPUT_DIR/'page_extraction_log.json').write_text(json.dumps(page_extraction_log,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
review.to_json(OUTPUT_DIR/'human_review_required.json',orient='records',indent=2,force_ascii=False)
assert list(orca_payload.columns)==['description','quantity','allowed']
print('Classification pages:',len(classification_df),'| Metadata records:',len(all_document_metadata),'| Line items:',len(line_items_df),'| Pet groups:',len(pet_claims))
zip_path=ROOT/'orca_classification_metadata_line_items_outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file(): z.write(p,arcname=str(p.relative_to(OUTPUT_DIR)))
files.download(str(zip_path))


Classification pages: 1 | Metadata records: 1 | Line items: 8 | Pet groups: 2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>